# POC03 — Rates Pressure × Market Resilience

## Research Goal
Test whether equity-market resilience under Treasury-rate pressure
helps distinguish benign rate repricing from regimes associated with
higher future equity risk.

## Stage A — Rates Pressure
Initial data source:
- DGS10: FRED / Federal Reserve H.15
- Frequency: Daily
- Unit: Percent
- Definition: 10-Year Treasury Constant Maturity Rate

## Research Rules
1. Define features before examining future equity labels.
2. Do not forward-fill missing macro observations without justification.
3. Keep feature construction point-in-time.
4. Separate hypothesis generation from confirmation.
5. Every feature must ultimately have a practical investment use.

In [1]:
!pip install pandas_datareader -q

import pandas as pd
import numpy as np
from pandas_datareader import data as pdr

In [2]:
# Download DGS10
start = "2004-01-01"
end = "2026-10-03"

dgs10 = pdr.DataReader(
    "DGS10",
    "fred",
    start,
    end
)

rates = dgs10.rename(columns={"DGS10": "US10Y"})

rates.tail(10)

,US10Y
DATE,
2026-09-21,4.96
2026-09-22,4.96
2026-09-23,5.11
2026-09-24,5.18
2026-09-25,5.17
2026-09-28,5.24
2026-09-29,5.26
2026-09-30,5.29
2026-10-01,5.24


In [3]:
# Data QA
print("Shape:", rates.shape)

print("\nFirst 5 rows:")
display(rates.head())

print("\nLast 10 rows:")
display(rates.tail(10))

print("\nMissing values:")
print(rates.isna().sum())

print("\nDate range:")
print("Start:", rates.index.min())
print("End:  ", rates.index.max())

print("\nSummary:")
display(rates.describe())

Shape: (5937, 1)

First 5 rows:


,US10Y
DATE,
2004-01-01,NaN
2004-01-02,4.38
2004-01-05,4.41
2004-01-06,4.29
2004-01-07,4.27



Last 10 rows:


,US10Y
DATE,
2026-09-21,4.96
2026-09-22,4.96
2026-09-23,5.11
2026-09-24,5.18
2026-09-25,5.17
2026-09-28,5.24
2026-09-29,5.26
2026-09-30,5.29
2026-10-01,5.24



Missing values:
US10Y    244
dtype: int64

Date range:
Start: 2004-01-01 00:00:00
End:   2026-10-02 00:00:00

Summary:


,US10Y
count,5693.000000
mean,3.079838
std,1.154808
min,0.520000
25%,2.130000
50%,2.970000
75%,4.160000
max,5.290000


In [4]:
# Deal with missing filed - NO filling missing observation
rates_clean = rates.dropna().copy()

print("Before:", len(rates))
print("After: ", len(rates_clean))
print("Dropped:", len(rates) - len(rates_clean))

rates_clean.tail()

Before: 5937
After:  5693
Dropped: 244


,US10Y
DATE,
2026-09-28,5.24
2026-09-29,5.26
2026-09-30,5.29
2026-10-01,5.24
2026-10-02,5.28


In [5]:
# Create 1st Feature: Level
window = 252

level_prior = rates_clean["US10Y"].shift(1).rolling(window)

rates_clean["LevelZ252"] = (
    (rates_clean["US10Y"] - level_prior.mean())
    / level_prior.std()
)

In [6]:
# QA
print(rates_clean[["US10Y", "LevelZ252"]].head(260))

print("\nLatest:")
display(
    rates_clean[
        ["US10Y", "LevelZ252"]
    ].tail(10)
)

print("\nSummary:")
display(
    rates_clean["LevelZ252"].describe()
)

            US10Y  LevelZ252
DATE                        
2004-01-02   4.38        NaN
2004-01-05   4.41        NaN
2004-01-06   4.29        NaN
2004-01-07   4.27        NaN
2004-01-08   4.27        NaN
...           ...        ...
2005-01-10   4.29   0.072287
2005-01-11   4.26  -0.038631
2005-01-12   4.25  -0.075359
2005-01-13   4.20  -0.261961
2005-01-14   4.23  -0.152665

[260 rows x 2 columns]

Latest:


,US10Y,LevelZ252
DATE,,
2026-09-21,4.96,2.514986
2026-09-22,4.96,2.476006
2026-09-23,5.11,3.032715
2026-09-24,5.18,3.241068
2026-09-25,5.17,3.125446
2026-09-28,5.24,3.319234
2026-09-29,5.26,3.310517
2026-09-30,5.29,3.336642
2026-10-01,5.24,3.075406



Summary:


,LevelZ252
count,5441.000000
mean,0.022887
std,1.450330
min,-4.727299
25%,-1.108171
50%,0.111807
75%,1.026141
max,4.109657


最直观定义：
\[
Momentum20_t=y_t-y_{t-20}
\]
但Treasury yield最好用 basis points：
\[
Momentum20BP_t=(y_t-y_{t-20})\times100
\]

In [7]:
# Create 2nd Feature: Momentum20
rates_clean["Momentum20BP"] = (
    rates_clean["US10Y"] - rates_clean["US10Y"].shift(20)
) * 100

display(
    rates_clean[
        ["US10Y", "LevelZ252", "Momentum20BP"]
    ].tail(25)
)

print("\nMomentum summary:")
display(rates_clean["Momentum20BP"].describe())

,US10Y,LevelZ252,Momentum20BP
DATE,,,
2026-08-28,4.73,2.063000,-2.0
2026-08-31,4.75,2.131269,5.0
2026-09-01,4.79,2.289799,16.0
2026-09-02,4.79,2.256532,16.0
2026-09-03,4.77,2.132939,8.0
2026-09-04,4.78,2.150515,13.0
2026-09-08,4.80,2.211591,8.0
2026-09-09,4.83,2.314614,13.0
2026-09-10,4.95,2.811434,27.0



Momentum summary:


,Momentum20BP
count,5673.000000
mean,0.316059
std,24.384143
min,-133.000000
25%,-14.000000
50%,0.000000
75%,15.000000
max,85.000000


In [8]:
# Create 3rd Feature 3：ShockZ60
# 先计算每天10Y变化，单位bp：
rates_clean["DailyChangeBP"] = (
    rates_clean["US10Y"] - rates_clean["US10Y"].shift(1)
) * 100

In [9]:
# 今天的变化，相对于此前60个Treasury observations的日变化有多异常
# DailyChangeBP = (US10Y_t - US10Y_t-1) * 100
shock_prior = (
    rates_clean["DailyChangeBP"]
    .shift(1)
    .rolling(60)
)

rates_clean["ShockZ60"] = (
    rates_clean["DailyChangeBP"] - shock_prior.mean()
) / shock_prior.std()

In [10]:
display(
    rates_clean[
        [
            "US10Y",
            "LevelZ252",
            "Momentum20BP",
            "DailyChangeBP",
            "ShockZ60"
        ]
    ].tail(25)
)

print("\nShock summary:")
display(rates_clean["ShockZ60"].describe())

,US10Y,LevelZ252,Momentum20BP,DailyChangeBP,ShockZ60
DATE,,,,,
2026-08-28,4.73,2.063000,-2.0,6.0,1.296980
2026-08-31,4.75,2.131269,5.0,2.0,0.363293
2026-09-01,4.79,2.289799,16.0,4.0,0.803490
2026-09-02,4.79,2.256532,16.0,0.0,-0.092790
2026-09-03,4.77,2.132939,8.0,-2.0,-0.552929
2026-09-04,4.78,2.150515,13.0,1.0,0.139568
2026-09-08,4.80,2.211591,8.0,2.0,0.376440
2026-09-09,4.83,2.314614,13.0,3.0,0.592790
2026-09-10,4.95,2.811434,27.0,12.0,2.800419



Shock summary:


,ShockZ60
count,5632.000000
mean,0.002285
std,1.040827
min,-5.689652
25%,-0.663170
50%,-0.018680
75%,0.663556
max,5.976705


In [11]:
# Create 4th Rates feature：Distance from High
# 先得到包含今天的60-observation high
rates_clean["High60"] = (
    rates_clean["US10Y"]
    .rolling(60)
    .max()
)

#然后算距离
rates_clean["DistanceHighBP"] = (
    rates_clean["US10Y"] - rates_clean["High60"]
) * 100

In [12]:
# 60个daily changes的波动率标准化
daily_vol_prior60 = (
    rates_clean["DailyChangeBP"]
    .shift(1)
    .rolling(60)
    .std()
)

rates_clean["DistanceHighVolAdj60"] = (
    rates_clean["DistanceHighBP"]
    / daily_vol_prior60
)

In [13]:
display(
    rates_clean[
        [
            "US10Y",
            "LevelZ252",
            "Momentum20BP",
            "DailyChangeBP",
            "ShockZ60",
            "High60",
            "DistanceHighBP",
            "DistanceHighVolAdj60"
        ]
    ].tail(25)
)

print("\nDistance-from-high summary:")
display(
    rates_clean[
        ["DistanceHighBP", "DistanceHighVolAdj60"]
    ].describe()
)

,US10Y,LevelZ252,Momentum20BP,DailyChangeBP,ShockZ60,High60,DistanceHighBP,DistanceHighVolAdj60
DATE,,,,,,,,
2026-08-28,4.73,2.063000,-2.0,6.0,1.296980,4.75,-2.0,-0.459108
2026-08-31,4.75,2.131269,5.0,2.0,0.363293,4.75,0.0,0.000000
2026-09-01,4.79,2.289799,16.0,4.0,0.803490,4.79,0.0,0.000000
2026-09-02,4.79,2.256532,16.0,0.0,-0.092790,4.79,0.0,0.000000
2026-09-03,4.77,2.132939,8.0,-2.0,-0.552929,4.79,-2.0,-0.463996
2026-09-04,4.78,2.150515,13.0,1.0,0.139568,4.79,-1.0,-0.232613
2026-09-08,4.80,2.211591,8.0,2.0,0.376440,4.80,0.0,0.000000
2026-09-09,4.83,2.314614,13.0,3.0,0.592790,4.83,0.0,0.000000
2026-09-10,4.95,2.811434,27.0,12.0,2.800419,4.95,0.0,0.000000



Distance-from-high summary:


,DistanceHighBP,DistanceHighVolAdj60
count,5634.000000,5632.000000
mean,-28.989173,-5.342048
std,26.894948,4.204908
min,-200.000000,-26.818128
25%,-42.000000,-7.986070
50%,-22.000000,-4.593899
75%,-9.000000,-1.913506
max,0.000000,0.000000


10/1 的状态可以准确写成：
Extreme High + Rising + Moderate Down Day + Still Near High

因为四个维度分别是：
- LevelZ252 = +3.08：相对过去一年，yield level 极高。
- Momentum20BP = +45bp：过去20个有效观测仍然明显上行。
- ShockZ60 = -1.35：当天 −5bp 是偏大的下跌，但远非极端shock。
- DistanceHighVolAdj60 = -1.09：离60-observation high只有5bp，约等于近期1.09个daily-vol单位，仍然贴着高位。

## Stage A — Rates Pressure Features (Frozen)

The Treasury-rate environment is represented using four separate features:

1. **LevelZ252**
   - Current DGS10 relative to the prior 252 valid Treasury observations.
   - Measures whether the current yield level is historically elevated or depressed.

2. **Momentum20BP**
   - Current DGS10 minus DGS10 20 valid observations earlier, expressed in basis points.
   - Measures the medium-term direction and magnitude of the yield move.

3. **ShockZ60**
   - Current daily DGS10 change relative to the prior 60 daily changes.
   - Measures whether today's rate move is unusually large.

4. **DistanceHighVolAdj60**
   - Distance from the rolling 60-observation high, normalized by prior 60-day yield-change volatility.
   - Measures whether yields remain near recent highs or have meaningfully retreated.

These features are intentionally kept separate.
No composite Rates Pressure score is constructed at this stage.

# POC03 preregistration
现在我们已经可以锁定：
Research question
Under elevated Treasury-rate pressure, does market resilience help distinguish regimes associated with greater forward equity downside?

# Primary label

\[
FAE_{20D,t}
=
\min_{k=1,\ldots,20}
\left(\frac{SPY_{t+k}}{SPY_t}-1\right)
\]

# Mechanism
High rates alone may not cause equity stress. Downside risk should become more severe when the market begins to lose its ability to absorb rates pressure, reflected in deterioration across price, breadth, credit, volatility and risk preference.

# Prediction — FROZEN
\[
\boxed{
High\ Rates\ Pressure
+
Weakening\ Market\ Resilience
\Rightarrow
More\ Negative\ FAE_{20D}
}
\]

# Stage B — Market Resilience

## B1. Price Resilience

The first Market Resilience block measures whether the equity market is
absorbing rates pressure at the headline index level.

Two dimensions are used:

1. **SPYMomentum20**
   - SPY's trailing 20-trading-day return.
   - Measures recent price direction.

2. **SPYDistanceHigh60**
   - SPY's distance from its rolling 60-trading-day high.
   - Measures accumulated price damage and whether the index remains near recent highs.

Both features use only information available at time t.

In [14]:
import yfinance as yf

start = "2004-01-01"
end = "2026-10-05"

spy_rsp = yf.download(
    ["SPY", "RSP"],
    start=start,
    end=end,
    auto_adjust=True,
    progress=False
)["Close"]

spy_rsp.tail(10)

Ticker,RSP,SPY
Date,,
2026-09-21,212.679993,773.500000
2026-09-22,212.800003,773.380005
2026-09-23,211.309998,767.809998
2026-09-24,210.259995,767.179993
2026-09-25,211.110001,771.349976
2026-09-28,209.740005,765.609985
2026-09-29,209.500000,764.200012
2026-09-30,208.020004,762.630005
2026-10-01,209.000000,763.989990


In [15]:
# QA
print("Shape:", spy_rsp.shape)
print("\nFirst rows:")
display(spy_rsp.head())

print("\nLast rows:")
display(spy_rsp.tail(10))

print("\nMissing values:")
print(spy_rsp.isna().sum())

print("\nDate range:")
print(spy_rsp.index.min(), "to", spy_rsp.index.max())

print("\nSummary:")
display(spy_rsp.describe())

Shape: (5724, 2)

First rows:


Ticker,RSP,SPY
Date,,
2004-01-02,23.677374,73.461761
2004-01-05,23.974201,74.260910
2004-01-06,24.023369,74.333572
2004-01-07,24.037441,74.584511
2004-01-08,24.220081,74.881721



Last rows:


Ticker,RSP,SPY
Date,,
2026-09-21,212.679993,773.500000
2026-09-22,212.800003,773.380005
2026-09-23,211.309998,767.809998
2026-09-24,210.259995,767.179993
2026-09-25,211.110001,771.349976
2026-09-28,209.740005,765.609985
2026-09-29,209.500000,764.200012
2026-09-30,208.020004,762.630005
2026-10-01,209.000000,763.989990



Missing values:
Ticker
RSP    0
SPY    0
dtype: int64

Date range:
2004-01-02 00:00:00 to 2026-10-02 00:00:00

Summary:


Ticker,RSP,SPY
count,5724.000000,5724.000000
mean,78.654952,233.644442
std,52.223265,177.061429
min,15.395429,49.557507
25%,33.750427,94.930368
50%,64.455185,169.294678
75%,119.359726,345.959442
max,221.935745,775.953186


In [16]:
# Step 2 — SPYMomentum20

price = spy_rsp.copy()

price["SPYMomentum20"] = (
    price["SPY"] / price["SPY"].shift(20) - 1
)

price[
    ["SPY", "SPYMomentum20"]
].tail(25)

Ticker,SPY,SPYMomentum20
Date,,
2026-08-28,767.444275,0.029878
2026-08-31,765.149963,0.012380
2026-09-01,759.893066,-0.012381
2026-09-02,763.264648,-0.006015
2026-09-03,771.254822,0.005998
2026-09-04,768.282227,-0.003970
2026-09-08,764.062683,-0.009146
2026-09-09,760.511536,-0.010590
2026-09-10,755.952820,-0.018978


In [17]:
# Step 3 - SPYDistanceHigh60

price["SPYHigh60"] = (
    price["SPY"]
    .rolling(60)
    .max()
)

price["SPYDistanceHigh60"] = (
    price["SPY"] / price["SPYHigh60"] - 1
)

price[
    [
        "SPY",
        "SPYMomentum20",
        "SPYHigh60",
        "SPYDistanceHigh60"
    ]
].tail(25)

Ticker,SPY,SPYMomentum20,SPYHigh60,SPYDistanceHigh60
Date,,,,
2026-08-28,767.444275,0.029878,775.953186,-0.010966
2026-08-31,765.149963,0.012380,775.953186,-0.013923
2026-09-01,759.893066,-0.012381,775.953186,-0.020697
2026-09-02,763.264648,-0.006015,775.953186,-0.016352
2026-09-03,771.254822,0.005998,775.953186,-0.006055
2026-09-04,768.282227,-0.003970,775.953186,-0.009886
2026-09-08,764.062683,-0.009146,775.953186,-0.015324
2026-09-09,760.511536,-0.010590,775.953186,-0.019900
2026-09-10,755.952820,-0.018978,775.953186,-0.025775


In [18]:
# Step 4 — QA
print("Missing values:")
print(
    price[
        ["SPYMomentum20", "SPYHigh60", "SPYDistanceHigh60"]
    ].isna().sum()
)

print("\nRecent values:")
display(
    price[
        ["SPY", "SPYMomentum20", "SPYHigh60", "SPYDistanceHigh60"]
    ].tail(25)
)

print("\nSummary:")
display(
    price[
        ["SPYMomentum20", "SPYDistanceHigh60"]
    ].describe()
)

Missing values:
Ticker
SPYMomentum20        20
SPYHigh60            59
SPYDistanceHigh60    59
dtype: int64

Recent values:


Ticker,SPY,SPYMomentum20,SPYHigh60,SPYDistanceHigh60
Date,,,,
2026-08-28,767.444275,0.029878,775.953186,-0.010966
2026-08-31,765.149963,0.012380,775.953186,-0.013923
2026-09-01,759.893066,-0.012381,775.953186,-0.020697
2026-09-02,763.264648,-0.006015,775.953186,-0.016352
2026-09-03,771.254822,0.005998,775.953186,-0.006055
2026-09-04,768.282227,-0.003970,775.953186,-0.009886
2026-09-08,764.062683,-0.009146,775.953186,-0.015324
2026-09-09,760.511536,-0.010590,775.953186,-0.019900
2026-09-10,755.952820,-0.018978,775.953186,-0.025775



Summary:


Ticker,SPYMomentum20,SPYDistanceHigh60
count,5704.000000,5665.000000
mean,0.009173,-0.031670
std,0.044014,0.047919
min,-0.309868,-0.417108
25%,-0.011072,-0.040696
50%,0.014742,-0.013354
75%,0.033801,-0.002102
max,0.236211,0.000000


In [19]:
# Step 5 Add breadth
# 先只建立 Relative Breadth，暂时不碰absolute breadth
price["RSPMomentum20"] = (
    price["RSP"] / price["RSP"].shift(20) - 1
)

price["BreadthMomentum20"] = (
    price["RSPMomentum20"] - price["SPYMomentum20"]
)

display(
    price[
        [
            "SPY",
            "RSP",
            "SPYMomentum20",
            "RSPMomentum20",
            "BreadthMomentum20"
        ]
    ].tail(25)
)

print("\nSummary:")
display(
    price[
        [
            "SPYMomentum20",
            "RSPMomentum20",
            "BreadthMomentum20"
        ]
    ].describe()
)

Ticker,SPY,RSP,SPYMomentum20,RSPMomentum20,BreadthMomentum20
Date,,,,,
2026-08-28,767.444275,219.863541,0.029878,0.026417,-0.003461
2026-08-31,765.149963,218.568405,0.012380,0.010502,-0.001878
2026-09-01,759.893066,216.775146,-0.012381,-0.011987,0.000394
2026-09-02,763.264648,217.781372,-0.006015,-0.005143,0.000872
2026-09-03,771.254822,219.225937,0.005998,0.006725,0.000727
2026-09-04,768.282227,218.179871,-0.003970,-0.004952,-0.000982
2026-09-08,764.062683,215.918365,-0.009146,-0.015848,-0.006702
2026-09-09,760.511536,213.836197,-0.010590,-0.027414,-0.016824
2026-09-10,755.952820,212.371704,-0.018978,-0.035779,-0.016801



Summary:


Ticker,SPYMomentum20,RSPMomentum20,BreadthMomentum20
count,5704.000000,5704.000000,5704.000000
mean,0.009173,0.008914,-0.000259
std,0.044014,0.049864,0.015263
min,-0.309868,-0.365283,-0.072118
25%,-0.011072,-0.014171,-0.007710
50%,0.014742,0.013583,-0.000337
75%,0.033801,0.035170,0.007210
max,0.236211,0.306694,0.095388


In [20]:
# change prior name market breadth to
# Equal-weight relative performance / market concentration
price["EqualWeightRelative20"] = (
    price["RSPMomentum20"] - price["SPYMomentum20"]
)

In [21]:
# result: 在整个 2004–2026 样本中，只有大约 1.77% 的交易日，RSP 相对 SPY 的 20 日表现比现在还差
# 也就是目前的 equal-weight underperformance / market concentration 已经处于历史左尾约 2%
# The divergence between the headline index and equal-weight stocks is historically extreme

latest_date = price.index.max()

latest_equal_weight_rel = price.loc[
    latest_date, "EqualWeightRelative20"
]

historical_percentile = (
    price["EqualWeightRelative20"]
    .dropna()
    .le(latest_equal_weight_rel)
    .mean()
)

print("Latest date:", latest_date)
print(
    "EqualWeightRelative20:",
    f"{latest_equal_weight_rel:.2%}"
)
print(
    "Historical percentile:",
    f"{historical_percentile:.2%}"
)

Latest date: 2026-10-02 00:00:00
EqualWeightRelative20: -4.12%
Historical percentile: 1.77%


## Stage B1 — Price & Concentration Features (Frozen)

The first Market Resilience block contains three features:

1. **SPYMomentum20**
   - SPY trailing 20-trading-day return.
   - Measures headline market direction.

2. **SPYDistanceHigh60**
   - SPY's percentage distance from its rolling 60-trading-day high.
   - Measures accumulated headline price damage.

3. **EqualWeightRelative20**
   - RSP trailing 20-day return minus SPY trailing 20-day return.
   - Measures equal-weight relative strength and serves as a proxy for market concentration.
   - Negative values indicate cap-weighted SPY is outperforming equal-weight RSP.

These features intentionally separate:
- Direction
- Price damage
- Internal concentration

EqualWeightRelative20 is not treated as a complete measure of market breadth.
Traditional breadth measures such as advance/decline data or percentage of stocks above moving averages are reserved for later research.

In [22]:
# Credit Block
hy_oas = pdr.DataReader(
    "BAMLH0A0HYM2",
    "fred",
    start,
    end
)

hy_oas = hy_oas.rename(
    columns={"BAMLH0A0HYM2": "HYOAS"}
)

print("Shape:", hy_oas.shape)

print("\nDate range:")
print(hy_oas.index.min(), "to", hy_oas.index.max())

print("\nMissing values:")
print(hy_oas.isna().sum())

print("\nRecent observations:")
display(hy_oas.tail(20))

print("\nSummary:")
display(hy_oas.describe())

Shape: (792, 1)

Date range:
2023-10-06 00:00:00 to 2026-10-02 00:00:00

Missing values:
HYOAS    8
dtype: int64

Recent observations:


,HYOAS
DATE,
2026-09-07,2.68
2026-09-08,2.67
2026-09-09,2.71
2026-09-10,2.70
2026-09-11,2.65
2026-09-14,2.71
2026-09-15,2.76
2026-09-16,2.70
2026-09-17,2.70



Summary:


,HYOAS
count,784.000000
mean,3.098278
std,0.403665
min,2.590000
25%,2.800000
50%,3.020000
75%,3.240000
max,4.610000


In [23]:
# QA

print("start =", start)
print("end =", end)

test_hy = pdr.DataReader(
    "BAMLH0A0HYM2",
    "fred",
    "2004-01-01",
    "2010-01-01"
)

print(test_hy.head())
print(test_hy.tail())
print(test_hy.shape)
print(test_hy.index.min(), test_hy.index.max())

start = 2004-01-01
end = 2026-10-05
Empty DataFrame
Columns: [BAMLH0A0HYM2]
Index: []
Empty DataFrame
Columns: [BAMLH0A0HYM2]
Index: []
(0, 1)
NaT NaT


### Credit Resilience — Data Availability Note

The preferred credit-risk measure is the ICE BofA US High Yield Index
Option-Adjusted Spread (BAMLH0A0HYM2).

As of April 2026, FRED restricts this series to approximately three years
of historical observations due to data availability/licensing constraints.

Because POC03 requires a long historical sample including major stress
episodes, the truncated FRED series will not be used as the primary
historical credit feature at this stage.

The conceptual credit features remain:

- HYOASLevel
- HYOASChange20BP

A suitable long-history credit dataset or proxy will be evaluated
separately before the Credit Resilience block is finalized.

In [24]:
# Wayback FRED snapshot：1996-12-31 → 2025-11-03

import pandas as pd

wayback_url = (
    "https://web.archive.org/web/20251104204105id_/"
    "https://fred.stlouisfed.org/graph/fredgraph.csv?id=BAMLH0A0HYM2"
)

hy_wayback = pd.read_csv(wayback_url)

print("Shape:", hy_wayback.shape)
print("\nColumns:")
print(hy_wayback.columns)

print("\nHead:")
display(hy_wayback.head())

print("\nTail:")
display(hy_wayback.tail())

Shape: (7622, 2)

Columns:
Index(['observation_date', 'BAMLH0A0HYM2'], dtype='object')

Head:


,observation_date,BAMLH0A0HYM2
0,1996-12-31,3.13
1,1997-01-01,NaN
2,1997-01-02,3.06
3,1997-01-03,3.09
4,1997-01-06,3.10



Tail:


,observation_date,BAMLH0A0HYM2
7617,2025-10-28,2.82
7618,2025-10-29,2.76
7619,2025-10-30,2.85
7620,2025-10-31,2.94
7621,2025-11-03,3.04


In [25]:
# Wayback data cleaning
hy_archive = hy_wayback.copy()

hy_archive["observation_date"] = pd.to_datetime(
    hy_archive["observation_date"]
)

hy_archive["BAMLH0A0HYM2"] = pd.to_numeric(
    hy_archive["BAMLH0A0HYM2"],
    errors="coerce"
)

hy_archive = (
    hy_archive
    .set_index("observation_date")
    .rename(columns={"BAMLH0A0HYM2": "HYOAS"})
    .dropna()
    .sort_index()
)

print("Archive shape:", hy_archive.shape)
print("Archive range:",
      hy_archive.index.min(),
      "to",
      hy_archive.index.max())

display(hy_archive.head())
display(hy_archive.tail())
display(hy_archive.describe())

Archive shape: (7530, 1)
Archive range: 1996-12-31 00:00:00 to 2025-11-03 00:00:00


,HYOAS
observation_date,
1996-12-31,3.13
1997-01-02,3.06
1997-01-03,3.09
1997-01-06,3.10
1997-01-07,3.10


,HYOAS
observation_date,
2025-10-28,2.82
2025-10-29,2.76
2025-10-30,2.85
2025-10-31,2.94
2025-11-03,3.04


,HYOAS
count,7530.000000
mean,5.227793
std,2.539372
min,2.410000
25%,3.540000
50%,4.560000
75%,6.180000
max,21.820000


In [26]:
# organize prior HY data
hy_current = hy_oas.copy()

hy_current["HYOAS"] = pd.to_numeric(
    hy_current["HYOAS"],
    errors="coerce"
)

hy_current = (
    hy_current
    .dropna()
    .sort_index()
)

print("Current FRED shape:", hy_current.shape)
print(
    "Current FRED range:",
    hy_current.index.min(),
    "to",
    hy_current.index.max()
)

Current FRED shape: (784, 1)
Current FRED range: 2023-10-06 00:00:00 to 2026-10-02 00:00:00


In [27]:
# Validate the overlapping period 2023-10-03 to 2025-11-03
overlap = hy_archive.join(
    hy_current,
    how="inner",
    lsuffix="_Archive",
    rsuffix="_Current"
)

overlap["Diff"] = (
    overlap["HYOAS_Archive"]
    - overlap["HYOAS_Current"]
)

print("Overlap observations:", len(overlap))

print(
    "Overlap range:",
    overlap.index.min(),
    "to",
    overlap.index.max()
)

print(
    "Max absolute difference:",
    overlap["Diff"].abs().max()
)

print(
    "Mean absolute difference:",
    overlap["Diff"].abs().mean()
)

print(
    "Number different > 0.01:",
    (overlap["Diff"].abs() > 0.01).sum()
)

display(overlap.tail(20))



Overlap observations: 543
Overlap range: 2023-10-06 00:00:00 to 2025-11-03 00:00:00
Max absolute difference: 0.0
Mean absolute difference: 0.0
Number different > 0.01: 0


,HYOAS_Archive,HYOAS_Current,Diff
2025-10-07,2.82,2.82,0.0
2025-10-08,2.84,2.84,0.0
2025-10-09,2.95,2.95,0.0
2025-10-10,3.18,3.18,0.0
2025-10-13,3.18,3.18,0.0
2025-10-14,3.11,3.11,0.0
2025-10-15,2.95,2.95,0.0
2025-10-16,3.04,3.04,0.0
2025-10-17,3.04,3.04,0.0
2025-10-20,2.99,2.99,0.0


In [28]:
# 拼成完整的 HY OAS
hy_full = pd.concat([
    hy_archive.loc[
        hy_archive.index < hy_current.index.min()
    ],
    hy_current
]).sort_index()

print("Full shape:", hy_full.shape)

print(
    "Full range:",
    hy_full.index.min(),
    "to",
    hy_full.index.max()
)

print("\nMissing values:")
print(hy_full.isna().sum())

print(
    "\nDuplicate dates:",
    hy_full.index.duplicated().sum()
)

print("\nSummary:")
display(hy_full.describe())

print("\nRecent observations:")
display(hy_full.tail(20))

Full shape: (7771, 1)
Full range: 1996-12-31 00:00:00 to 2026-10-02 00:00:00

Missing values:
HYOAS    0
dtype: int64

Duplicate dates: 0

Summary:


,HYOAS
count,7771.000000
mean,5.154362
std,2.533345
min,2.410000
25%,3.450000
50%,4.490000
75%,6.130000
max,21.820000



Recent observations:


,HYOAS
2026-09-07,2.68
2026-09-08,2.67
2026-09-09,2.71
2026-09-10,2.70
2026-09-11,2.65
2026-09-14,2.71
2026-09-15,2.76
2026-09-16,2.70
2026-09-17,2.70
2026-09-18,2.68


In [29]:
credit = hy_full.loc["2004-01-01":].copy()

credit["HYOASLevel"] = credit["HYOAS"]

credit["HYOASChange20BP"] = (
    credit["HYOAS"]
    - credit["HYOAS"].shift(20)
) * 100

In [30]:
# QA
print("Credit shape:", credit.shape)

print("\nMissing values:")
print(
    credit[
        ["HYOASLevel", "HYOASChange20BP"]
    ].isna().sum()
)

print("\nRecent observations:")
display(
    credit[
        ["HYOASLevel", "HYOASChange20BP"]
    ].tail(25)
)

print("\nSummary:")
display(
    credit[
        ["HYOASLevel", "HYOASChange20BP"]
    ].describe()
)

Credit shape: (5945, 3)

Missing values:
HYOASLevel          0
HYOASChange20BP    20
dtype: int64

Recent observations:


,HYOASLevel,HYOASChange20BP
2026-08-31,2.63,-15.0
2026-09-01,2.65,-8.0
2026-09-02,2.66,-9.0
2026-09-03,2.65,-6.0
2026-09-04,2.68,-2.0
2026-09-07,2.68,-2.0
2026-09-08,2.67,-5.0
2026-09-09,2.71,0.0
2026-09-10,2.70,-1.0
2026-09-11,2.65,-2.0



Summary:


,HYOASLevel,HYOASChange20BP
count,5945.000000,5925.000000
mean,4.916700,-0.376034
std,2.568774,76.153970
min,2.410000,-496.000000
25%,3.430000,-27.000000
50%,4.150000,-7.000000
75%,5.540000,21.000000
max,21.820000,682.000000


## Stage B2 — Credit Resilience Features (Frozen)

Credit resilience is represented using the ICE BofA US High Yield
Index Option-Adjusted Spread (BAMLH0A0HYM2).

Historical observations were recovered from an archived FRED CSV
snapshot and validated against 546 overlapping observations from the
current FRED series between 2023-10-03 and 2025-11-03.

The overlap matched exactly:
- Maximum absolute difference: 0.00
- Mean absolute difference: 0.00

Two credit features are retained:

1. HYOASLevel
   - Current HY option-adjusted spread, in percent.
   - Measures the absolute level of credit stress.

2. HYOASChange20BP
   - Change in HY OAS over the prior 20 valid observations, in basis points.
   - Measures the direction and speed of credit deterioration or improvement.

Level and direction are intentionally kept separate.
No fixed credit-stress threshold is imposed.

# VIX Block

In [31]:
# VIX block

vix = yf.download(
    "^VIX",
    start="2004-01-01",
    end="2026-10-05",
    auto_adjust=False,
    progress=False
)

vix.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,,
2004-01-02,18.219999,18.219999,18.68,17.540001,17.959999,0
2004-01-05,17.490000,17.490000,18.49,17.440001,18.450001,0
2004-01-06,16.730000,16.730000,17.67,16.190001,17.660000,0
2004-01-07,15.500000,15.500000,16.75,15.500000,16.719999,0
2004-01-08,15.610000,15.610000,15.68,15.320000,15.420000,0


In [32]:
print(vix.columns)
print(vix.shape)
display(vix.head())
display(vix.tail())

MultiIndex([('Adj Close', '^VIX'),
            (    'Close', '^VIX'),
            (     'High', '^VIX'),
            (      'Low', '^VIX'),
            (     'Open', '^VIX'),
            (   'Volume', '^VIX')],
           names=['Price', 'Ticker'])
(5726, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,,
2004-01-02,18.219999,18.219999,18.68,17.540001,17.959999,0
2004-01-05,17.490000,17.490000,18.49,17.440001,18.450001,0
2004-01-06,16.730000,16.730000,17.67,16.190001,17.660000,0
2004-01-07,15.500000,15.500000,16.75,15.500000,16.719999,0
2004-01-08,15.610000,15.610000,15.68,15.320000,15.420000,0


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,,
2026-09-28,16.070000,16.070000,16.620001,15.680000,16.16,0
2026-09-29,16.040001,16.040001,16.440001,15.730000,16.17,0
2026-09-30,16.340000,16.340000,16.610001,15.620000,15.93,0
2026-10-01,16.389999,16.389999,17.590000,16.209999,16.33,0
2026-10-02,15.310000,15.310000,16.240000,15.300000,16.15,0


In [33]:
vix_close = vix["Close"].copy()

# If yfinance returns a one-column DataFrame
if isinstance(vix_close, pd.DataFrame):
    vix_close = vix_close.iloc[:, 0]

vix_df = pd.DataFrame({
    "VIX": vix_close
})

print("Shape:", vix_df.shape)

print("\nDate range:")
print(vix_df.index.min(), "to", vix_df.index.max())

print("\nMissing values:")
print(vix_df.isna().sum())

print("\nRecent observations:")
display(vix_df.tail(20))

print("\nSummary:")
display(vix_df.describe())

Shape: (5726, 1)

Date range:
2004-01-02 00:00:00 to 2026-10-02 00:00:00

Missing values:
VIX    0
dtype: int64

Recent observations:


,VIX
Date,
2026-09-07,15.300000
2026-09-08,15.720000
2026-09-09,16.459999
2026-09-10,17.840000
2026-09-11,15.840000
2026-09-14,17.100000
2026-09-15,17.200001
2026-09-16,17.709999
2026-09-17,15.440000



Summary:


,VIX
count,5726.000000
mean,18.952548
std,8.375800
min,9.140000
25%,13.570000
50%,16.600000
75%,21.600000
max,82.690002


In [34]:
# Step 1：建立 VIX Level + Direction
vix_df["VIXLevel"] = vix_df["VIX"]

vix_df["VIXChange20"] = (
    vix_df["VIX"] / vix_df["VIX"].shift(20) - 1
)

print("Missing values:")
print(
    vix_df[
        ["VIXLevel", "VIXChange20"]
    ].isna().sum()
)

print("\nRecent values:")
display(
    vix_df[
        ["VIX", "VIXChange20"]
    ].tail(25)
)

print("\nSummary:")
display(
    vix_df[
        ["VIXLevel", "VIXChange20"]
    ].describe()
)

Missing values:
VIXLevel        0
VIXChange20    20
dtype: int64

Recent values:


,VIX,VIXChange20
Date,,
2026-08-31,14.920000,-0.059269
2026-09-01,16.340000,-0.009697
2026-09-02,15.200000,-0.038583
2026-09-03,14.320000,-0.054785
2026-09-04,14.530000,-0.024832
2026-09-07,15.300000,-0.010349
2026-09-08,15.720000,0.028796
2026-09-09,16.459999,0.131271
2026-09-10,17.840000,0.219412



Summary:


,VIXLevel,VIXChange20
count,5726.000000,5706.000000
mean,18.952548,0.031032
std,8.375800,0.302095
min,9.140000,-0.562463
25%,13.570000,-0.130207
50%,16.600000,-0.019822
75%,21.600000,0.123252
max,82.690002,5.044591


## Stage B2 — Volatility Resilience Features (Frozen)

Equity volatility resilience is represented using two separate features:

1. **VIXLevel**
   - Current VIX level.
   - Measures the current level of implied equity-market volatility.

2. **VIXChange20**
   - Percentage change in VIX over the prior 20 trading days.
   - Measures whether implied volatility is being repriced higher or lower.

These features are intentionally kept separate.

A low VIX level does not necessarily imply improving conditions if VIX is
rising rapidly, while a high VIX level may coexist with improving conditions
if volatility is normalizing.

No fixed VIX threshold (e.g. VIX < 20) is used.

In [35]:
# Calendar check for Rate, Breadth, and Volitality
print("Rates observations:", len(rates_clean))
print("Price observations:", len(price))
print("VIX observations:", len(vix_df))

print("\nRates range:")
print(rates_clean.index.min(), rates_clean.index.max())

print("\nPrice range:")
print(price.index.min(), price.index.max())

print("\nVIX range:")
print(vix_df.index.min(), vix_df.index.max())

# Dates present in SPY but missing from DGS10
spy_not_rates = price.index.difference(rates_clean.index)

# Dates present in DGS10 but missing from SPY
rates_not_spy = rates_clean.index.difference(price.index)

# Dates present in VIX but missing from SPY
vix_not_spy = vix_df.index.difference(price.index)

print("\nSPY dates missing from DGS10:")
print(len(spy_not_rates))
print(spy_not_rates[-20:])

print("\nDGS10 dates missing from SPY:")
print(len(rates_not_spy))
print(rates_not_spy[-20:])

print("\nVIX dates missing from SPY:")
print(len(vix_not_spy))
print(vix_not_spy[-20:])

Rates observations: 5693
Price observations: 5724
VIX observations: 5726

Rates range:
2004-01-02 00:00:00 2026-10-02 00:00:00

Price range:
2004-01-02 00:00:00 2026-10-02 00:00:00

VIX range:
2004-01-02 00:00:00 2026-10-02 00:00:00

SPY dates missing from DGS10:
41
DatetimeIndex(['2015-10-12', '2015-11-11', '2016-10-10', '2016-11-11',
               '2017-10-09', '2018-10-08', '2018-11-12', '2019-10-14',
               '2019-11-11', '2020-10-12', '2020-11-11', '2021-10-11',
               '2021-11-11', '2022-10-10', '2022-11-11', '2023-10-09',
               '2024-10-14', '2024-11-11', '2025-10-13', '2025-11-11'],
              dtype='datetime64[ns]', freq=None)

DGS10 dates missing from SPY:
10
DatetimeIndex(['2007-01-02', '2007-04-06', '2010-04-02', '2012-04-06',
               '2012-10-29', '2015-04-03', '2021-04-02', '2023-04-07',
               '2025-01-09', '2026-04-03'],
              dtype='datetime64[ns]', freq=None)

VIX dates missing from SPY:
2
DatetimeIndex(['2026-05-25',

Use only dates on which Rates + SPY/RSP + VIX all have observed values.
At each observation date \(t\), every feature represents an actual observation dated \(t\).

# Step 1：建立 Master Feature Table

In [36]:
# Stage A — Rates Pressure
rates_features = rates_clean[
    [
        "LevelZ252",
        "Momentum20BP",
        "ShockZ60",
        "DistanceHighVolAdj60"
    ]
].copy()

# Stage B — Price & Concentration
market_features = price[
    [
        "SPY",
        "SPYMomentum20",
        "SPYDistanceHigh60",
        "EqualWeightRelative20"
    ]
].copy()

# Stage B — Volatility
vol_features = vix_df[
    [
        "VIXLevel",
        "VIXChange20"
    ]
].copy()

# Inner join = intersection of observed dates
features = (
    rates_features
    .join(market_features, how="inner")
    .join(vol_features, how="inner")
)

print("Shape:", features.shape)

print("\nDate range:")
print(features.index.min(), "to", features.index.max())

print("\nMissing values:")
print(features.isna().sum())

print("\nRecent observations:")
display(features.tail(10))

Shape: (5683, 10)

Date range:
2004-01-02 00:00:00 to 2026-10-02 00:00:00

Missing values:
LevelZ252                252
Momentum20BP              20
ShockZ60                  61
DistanceHighVolAdj60      61
SPY                        0
SPYMomentum20             20
SPYDistanceHigh60         59
EqualWeightRelative20     20
VIXLevel                   0
VIXChange20               20
dtype: int64

Recent observations:


,LevelZ252,Momentum20BP,ShockZ60,DistanceHighVolAdj60,SPY,SPYMomentum20,SPYDistanceHigh60,EqualWeightRelative20,VIXLevel,VIXChange20
2026-09-21,2.514986,22.0,-1.438810,-1.199009,773.500000,0.012669,-0.003162,-0.049618,14.870000,-0.061830
2026-09-22,2.476006,26.0,-0.220432,-1.180886,773.380005,0.015496,-0.003316,-0.053030,14.210000,-0.080259
2026-09-23,3.032715,47.0,3.326641,0.000000,767.809998,0.004970,-0.010494,-0.048554,15.180000,-0.001972
2026-09-24,3.241068,52.0,1.260409,0.000000,767.179993,0.003923,-0.011306,-0.053716,15.670000,0.079945
2026-09-25,3.125446,50.0,-0.484675,-0.217018,771.349976,0.002808,-0.005932,-0.045917,14.870000,0.030492
2026-09-28,3.319234,51.0,1.271123,0.000000,765.609985,-0.002390,-0.013330,-0.043655,16.070000,0.077078
2026-09-29,3.310517,51.0,0.160817,0.000000,764.200012,-0.001242,-0.015147,-0.040248,16.040001,-0.018360
2026-09-30,3.336642,50.0,0.365180,0.000000,762.630005,0.003602,-0.017170,-0.043990,16.340000,0.075000
2026-10-01,3.075406,45.0,-1.354913,-1.086829,763.989990,0.000950,-0.015417,-0.041272,16.389999,0.144553
2026-10-02,3.147044,51.0,0.613798,-0.214115,769.640015,-0.002094,-0.008136,-0.041222,15.310000,0.053682


### Calendar Alignment Rule

The initial POC uses the intersection of dates for which DGS10,
SPY/RSP, and VIX all have observed values.

No forward-filling is applied.

This deliberately sacrifices a small number of observations in exchange
for a simpler and more transparent point-in-time design.

The prediction timestamp is assumed to be after the close of date t.
A more precise release-time/as-of treatment of macro data is reserved
for future production-level research.

In [37]:
# Combine 4 blocks
# ① Rates Pressure → rates_clean
# ② Price & Concentration → price
# ③ Credit Resilience → credit
# ④ Volatility Resilience → vix_df
# only use inner join / observed-date intersection，No ffill()

# ============================================================
# POC03 — Master Feature Table
# Four frozen blocks
# ============================================================

# 1. Rates Pressure
rates_features = rates_clean[
    [
        "LevelZ252",
        "Momentum20BP",
        "ShockZ60",
        "DistanceHighVolAdj60"
    ]
].copy()


# 2. Price & Concentration
price_features = price[
    [
        "SPY",
        "SPYMomentum20",
        "SPYDistanceHigh60",
        "EqualWeightRelative20"
    ]
].copy()


# 3. Credit Resilience
credit_features = credit[
    [
        "HYOASLevel",
        "HYOASChange20BP"
    ]
].copy()


# 4. Volatility Resilience
vol_features = vix_df[
    [
        "VIXLevel",
        "VIXChange20"
    ]
].copy()

In [38]:
# Combine using shared dates

master = (
    rates_features
    .join(price_features, how="inner")
    .join(credit_features, how="inner")
    .join(vol_features, how="inner")
    .sort_index()
)

print("Master shape:", master.shape)

print("\nDate range:")
print(master.index.min(), "to", master.index.max())

print("\nMissing values:")
print(master.isna().sum())

print("\nDuplicate dates:")
print(master.index.duplicated().sum())

print("\nRecent observations:")
display(master.tail(10))

Master shape: (5681, 12)

Date range:
2004-01-02 00:00:00 to 2026-10-02 00:00:00

Missing values:
LevelZ252                252
Momentum20BP              20
ShockZ60                  61
DistanceHighVolAdj60      61
SPY                        0
SPYMomentum20             20
SPYDistanceHigh60         59
EqualWeightRelative20     20
HYOASLevel                 0
HYOASChange20BP           19
VIXLevel                   0
VIXChange20               20
dtype: int64

Duplicate dates:
0

Recent observations:


,LevelZ252,Momentum20BP,ShockZ60,DistanceHighVolAdj60,SPY,SPYMomentum20,SPYDistanceHigh60,EqualWeightRelative20,HYOASLevel,HYOASChange20BP,VIXLevel,VIXChange20
2026-09-21,2.514986,22.0,-1.438810,-1.199009,773.500000,0.012669,-0.003162,-0.049618,2.66,-3.0,14.870000,-0.061830
2026-09-22,2.476006,26.0,-0.220432,-1.180886,773.380005,0.015496,-0.003316,-0.053030,2.68,-2.0,14.210000,-0.080259
2026-09-23,3.032715,47.0,3.326641,0.000000,767.809998,0.004970,-0.010494,-0.048554,2.73,6.0,15.180000,-0.001972
2026-09-24,3.241068,52.0,1.260409,0.000000,767.179993,0.003923,-0.011306,-0.053716,2.80,17.0,15.670000,0.079945
2026-09-25,3.125446,50.0,-0.484675,-0.217018,771.349976,0.002808,-0.005932,-0.045917,2.93,33.0,14.870000,0.030492
2026-09-28,3.319234,51.0,1.271123,0.000000,765.609985,-0.002390,-0.013330,-0.043655,3.02,39.0,16.070000,0.077078
2026-09-29,3.310517,51.0,0.160817,0.000000,764.200012,-0.001242,-0.015147,-0.040248,3.08,43.0,16.040001,-0.018360
2026-09-30,3.336642,50.0,0.365180,0.000000,762.630005,0.003602,-0.017170,-0.043990,3.12,46.0,16.340000,0.075000
2026-10-01,3.075406,45.0,-1.354913,-1.086829,763.989990,0.000950,-0.015417,-0.041272,3.24,59.0,16.389999,0.144553
2026-10-02,3.147044,51.0,0.613798,-0.214115,769.640015,-0.002094,-0.008136,-0.041222,3.10,42.0,15.310000,0.053682


## Master Feature Table — Four Core Blocks

The four core feature blocks are aligned using the intersection of
observed dates. No forward filling is applied.

Feature calculations are performed on each source's native observation
calendar before calendar alignment.

The Master Table contains:

### Rates Pressure
- LevelZ252
- Momentum20BP
- ShockZ60
- DistanceHighVolAdj60

### Price & Concentration
- SPYMomentum20
- SPYDistanceHigh60
- EqualWeightRelative20

### Credit Resilience
- HYOASLevel
- HYOASChange20BP

### Volatility Resilience
- VIXLevel
- VIXChange20

SPY is retained as the future label source and is not treated as a feature.

Initial missing values caused by rolling-window construction are retained
at this stage and will be removed only when the final modeling dataset is built.

In [39]:
# Check how many dates each block has lost
print("Individual block sizes:")
print("Rates :", len(rates_features))
print("Price :", len(price_features))
print("Credit:", len(credit_features))
print("VIX   :", len(vol_features))
print("Master:", len(master))

print("\nDates lost relative to SPY calendar:")
print(len(price_features.index.difference(master.index)))

print("\nMost recent SPY dates excluded from Master:")
print(
    price_features.index
    .difference(master.index)[-20:]
)

Individual block sizes:
Rates : 5693
Price : 5724
Credit: 5945
VIX   : 5726
Master: 5681

Dates lost relative to SPY calendar:
43

Most recent SPY dates excluded from Master:
DatetimeIndex(['2015-10-12', '2015-11-11', '2016-10-10', '2016-11-11',
               '2017-10-09', '2018-10-08', '2018-11-12', '2019-10-14',
               '2019-11-11', '2020-10-12', '2020-11-11', '2021-10-11',
               '2021-11-11', '2022-10-10', '2022-11-11', '2023-10-09',
               '2024-10-14', '2024-11-11', '2025-10-13', '2025-11-11'],
              dtype='datetime64[ns]', freq=None)


In [40]:
# Confirm all 4 blocks are included
print(master.columns.tolist())

['LevelZ252', 'Momentum20BP', 'ShockZ60', 'DistanceHighVolAdj60', 'SPY', 'SPYMomentum20', 'SPYDistanceHigh60', 'EqualWeightRelative20', 'HYOASLevel', 'HYOASChange20BP', 'VIXLevel', 'VIXChange20']


### Calendar Alignment Rule

The four feature blocks use different native observation calendars.

The Master Feature Table retains only dates for which all four blocks
have observed data (`inner join`). No forward filling is applied.

Feature construction is performed on each source's native calendar
before alignment.

Relative to the SPY trading calendar, 44 of 5,724 dates are excluded
(~0.77%). Many excluded dates correspond to U.S. equity trading days
without corresponding Treasury/FRED observations.

This small sample loss is accepted in order to avoid introducing stale
or artificially forward-filled macro observations.

# Add Cross-Asset / FX USDJPY and DXY to feature

In [41]:
import yfinance as yf
import pandas as pd
import numpy as np

start = "2004-01-01"
end = "2026-10-05"

fx_raw = yf.download(
    ["JPY=X", "DX-Y.NYB"],
    start=start,
    end=end,
    auto_adjust=True,
    progress=False
)["Close"]

fx_raw = fx_raw.rename(columns={
    "JPY=X": "USDJPY",
    "DX-Y.NYB": "DXY"
}).sort_index()

print("Shape:", fx_raw.shape)

print("\nDate range:")
print(fx_raw.index.min(), "to", fx_raw.index.max())

print("\nMissing values:")
print(fx_raw.isna().sum())

print("\nHead:")
display(fx_raw.head(10))

print("\nTail:")
display(fx_raw.tail(10))

print("\nSummary:")
display(fx_raw.describe())

Shape: (5934, 2)

Date range:
2004-01-01 00:00:00 to 2026-10-05 00:00:00

Missing values:
Ticker
DXY       196
USDJPY     31
dtype: int64

Head:


Ticker,DXY,USDJPY
Date,,
2004-01-01,NaN,107.300003
2004-01-02,86.919998,106.910004
2004-01-05,86.070000,106.239998
2004-01-06,85.629997,106.230003
2004-01-07,86.199997,106.180000
2004-01-08,85.570000,106.169998
2004-01-09,85.110001,106.580002
2004-01-12,85.550003,106.599998
2004-01-13,85.510002,106.239998



Tail:


Ticker,DXY,USDJPY
Date,,
2026-09-22,100.599998,157.369003
2026-09-23,101.099998,157.464005
2026-09-24,101.290001,158.264999
2026-09-25,100.970001,158.811005
2026-09-28,101.199997,157.462997
2026-09-29,101.370003,157.360992
2026-09-30,101.449997,157.404007
2026-10-01,102.099998,157.557999
2026-10-02,101.930000,157.927002



Summary:


Ticker,DXY,USDJPY
count,5738.000000,5903.000000
mean,90.435819,112.974630
std,9.439842,20.712908
min,71.330002,75.739998
25%,81.750000,102.176003
50%,90.720001,109.919998
75%,97.980003,119.796001
max,114.110001,163.863998


In [42]:
# Separate the two FX series first
dxy = fx_raw[["DXY"]].dropna().copy()
usdjpy = fx_raw[["USDJPY"]].dropna().copy()

print("DXY:")
print("Observations:", len(dxy))
print("Range:", dxy.index.min(), "to", dxy.index.max())

print("\nUSDJPY:")
print("Observations:", len(usdjpy))
print("Range:", usdjpy.index.min(), "to", usdjpy.index.max())

DXY:
Observations: 5738
Range: 2004-01-02 00:00:00 to 2026-10-05 00:00:00

USDJPY:
Observations: 5903
Range: 2004-01-01 00:00:00 to 2026-10-02 00:00:00


In [43]:
dxy_weekend = dxy[dxy.index.dayofweek >= 5]
usdjpy_weekend = usdjpy[usdjpy.index.dayofweek >= 5]

print("DXY weekend observations:", len(dxy_weekend))
display(dxy_weekend.tail(20))

print("\nUSDJPY weekend observations:", len(usdjpy_weekend))
display(usdjpy_weekend.tail(20))

DXY weekend observations: 0


Ticker,DXY
Date,



USDJPY weekend observations: 0


Ticker,USDJPY
Date,


In [44]:
# Remove non-weekday observations before feature construction
dxy = dxy[dxy.index.dayofweek < 5].copy()
usdjpy = usdjpy[usdjpy.index.dayofweek < 5].copy()

# Calculate 20-observation changes on each native valid calendar
dxy["DXYChange20"] = (
    dxy["DXY"] / dxy["DXY"].shift(20) - 1
)

usdjpy["USDJPYChange20"] = (
    usdjpy["USDJPY"] / usdjpy["USDJPY"].shift(20) - 1
)

In [45]:
# QA
print("DXY missing:")
print(dxy.isna().sum())

print("\nUSDJPY missing:")
print(usdjpy.isna().sum())

print("\nDXY summary:")
display(dxy[["DXY", "DXYChange20"]].describe())

print("\nUSDJPY summary:")
display(usdjpy[["USDJPY", "USDJPYChange20"]].describe())

print("\nRecent DXY:")
display(dxy.tail(25))

print("\nRecent USDJPY:")
display(usdjpy.tail(25))

DXY missing:
Ticker
DXY             0
DXYChange20    20
dtype: int64

USDJPY missing:
Ticker
USDJPY             0
USDJPYChange20    20
dtype: int64

DXY summary:


Ticker,DXY,DXYChange20
count,5738.000000,5718.000000
mean,90.435819,0.000749
std,9.439842,0.021000
min,71.330002,-0.096830
25%,81.750000,-0.012600
50%,90.720001,0.000390
75%,97.980003,0.013653
max,114.110001,0.123327



USDJPY summary:


Ticker,USDJPY,USDJPYChange20
count,5903.000000,5883.000000
mean,112.974630,0.001652
std,20.712908,0.026453
min,75.739998,-0.146677
25%,102.176003,-0.014246
50%,109.919998,0.001866
75%,119.796001,0.016991
max,163.863998,0.115035



Recent DXY:


Ticker,DXY,DXYChange20
Date,,
2026-08-31,99.430000,-0.005302
2026-09-01,99.669998,-0.002202
2026-09-02,99.559998,-0.001304
2026-09-03,99.000000,-0.009703
2026-09-04,99.160004,-0.004418
2026-09-08,98.839996,-0.009718
2026-09-09,98.769997,-0.010519
2026-09-10,99.089996,-0.009199
2026-09-11,99.120003,-0.008403



Recent USDJPY:


Ticker,USDJPY,USDJPYChange20
Date,,
2026-08-31,160.121994,0.016119
2026-09-01,159.746994,0.014080
2026-09-02,160.195999,0.015879
2026-09-03,158.923004,0.008395
2026-09-04,155.660004,-0.017354
2026-09-07,156.197006,-0.010729
2026-09-08,153.854996,-0.033307
2026-09-09,153.477997,-0.036336
2026-09-10,153.572998,-0.036120


# Combine FX to Master

In [46]:
fx_features = (
    dxy[["DXYChange20"]]
    .join(
        usdjpy[["USDJPYChange20"]],
        how="inner"
    )
)

print("FX shape:", fx_features.shape)
print(
    "FX range:",
    fx_features.index.min(),
    "to",
    fx_features.index.max()
)

print("\nMissing:")
print(fx_features.isna().sum())

display(fx_features.tail(10))

FX shape: (5707, 2)
FX range: 2004-01-02 00:00:00 to 2026-10-02 00:00:00

Missing:
Ticker
DXYChange20       20
USDJPYChange20    19
dtype: int64


Ticker,DXYChange20,USDJPYChange20
Date,,
2026-09-21,0.016498,-0.011693
2026-09-22,0.016162,-0.011122
2026-09-23,0.022038,-0.011047
2026-09-24,0.021377,-0.006216
2026-09-25,0.018253,-0.003201
2026-09-28,0.015045,-0.016606
2026-09-29,0.019511,-0.014936
2026-09-30,0.017859,-0.017429
2026-10-01,0.025512,-0.008589


In [47]:
master_final = (
    master
    .join(fx_features, how="inner")
    .sort_index()
)

print("Final Master shape:", master_final.shape)

print("\nDate range:")
print(
    master_final.index.min(),
    "to",
    master_final.index.max()
)

print("\nMissing values:")
print(master_final.isna().sum())

print("\nDuplicate dates:")
print(master_final.index.duplicated().sum())

print("\nColumns:")
print(master_final.columns.tolist())

print("\nRecent observations:")
display(master_final.tail(10))

Final Master shape: (5651, 14)

Date range:
2004-01-02 00:00:00 to 2026-10-02 00:00:00

Missing values:
LevelZ252                251
Momentum20BP              20
ShockZ60                  61
DistanceHighVolAdj60      61
SPY                        0
SPYMomentum20             20
SPYDistanceHigh60         59
EqualWeightRelative20     20
HYOASLevel                 0
HYOASChange20BP           19
VIXLevel                   0
VIXChange20               20
DXYChange20               19
USDJPYChange20            18
dtype: int64

Duplicate dates:
0

Columns:
['LevelZ252', 'Momentum20BP', 'ShockZ60', 'DistanceHighVolAdj60', 'SPY', 'SPYMomentum20', 'SPYDistanceHigh60', 'EqualWeightRelative20', 'HYOASLevel', 'HYOASChange20BP', 'VIXLevel', 'VIXChange20', 'DXYChange20', 'USDJPYChange20']

Recent observations:


,LevelZ252,Momentum20BP,ShockZ60,DistanceHighVolAdj60,SPY,SPYMomentum20,SPYDistanceHigh60,EqualWeightRelative20,HYOASLevel,HYOASChange20BP,VIXLevel,VIXChange20,DXYChange20,USDJPYChange20
2026-09-21,2.514986,22.0,-1.438810,-1.199009,773.500000,0.012669,-0.003162,-0.049618,2.66,-3.0,14.870000,-0.061830,0.016498,-0.011693
2026-09-22,2.476006,26.0,-0.220432,-1.180886,773.380005,0.015496,-0.003316,-0.053030,2.68,-2.0,14.210000,-0.080259,0.016162,-0.011122
2026-09-23,3.032715,47.0,3.326641,0.000000,767.809998,0.004970,-0.010494,-0.048554,2.73,6.0,15.180000,-0.001972,0.022038,-0.011047
2026-09-24,3.241068,52.0,1.260409,0.000000,767.179993,0.003923,-0.011306,-0.053716,2.80,17.0,15.670000,0.079945,0.021377,-0.006216
2026-09-25,3.125446,50.0,-0.484675,-0.217018,771.349976,0.002808,-0.005932,-0.045917,2.93,33.0,14.870000,0.030492,0.018253,-0.003201
2026-09-28,3.319234,51.0,1.271123,0.000000,765.609985,-0.002390,-0.013330,-0.043655,3.02,39.0,16.070000,0.077078,0.015045,-0.016606
2026-09-29,3.310517,51.0,0.160817,0.000000,764.200012,-0.001242,-0.015147,-0.040248,3.08,43.0,16.040001,-0.018360,0.019511,-0.014936
2026-09-30,3.336642,50.0,0.365180,0.000000,762.630005,0.003602,-0.017170,-0.043990,3.12,46.0,16.340000,0.075000,0.017859,-0.017429
2026-10-01,3.075406,45.0,-1.354913,-1.086829,763.989990,0.000950,-0.015417,-0.041272,3.24,59.0,16.389999,0.144553,0.025512,-0.008589
2026-10-02,3.147044,51.0,0.613798,-0.214115,769.640015,-0.002094,-0.008136,-0.041222,3.10,42.0,15.310000,0.053682,0.029596,0.014564


In [48]:
# QA
print(
    "Core Master observations:",
    len(master)
)

print(
    "Final Master observations:",
    len(master_final)
)

print(
    "Dates lost after adding FX:",
    len(master.index.difference(master_final.index))
)

print("\nMost recent lost dates:")
print(
    master.index
    .difference(master_final.index)[-20:]
)

Core Master observations: 5681
Final Master observations: 5651
Dates lost after adding FX: 30

Most recent lost dates:
DatetimeIndex(['2008-08-05', '2008-08-06', '2008-08-07', '2008-08-11',
               '2008-08-12', '2008-08-13', '2008-08-14', '2008-08-15',
               '2008-08-18', '2008-08-19', '2008-08-20', '2008-08-21',
               '2008-08-22', '2008-08-25', '2011-04-15', '2013-10-08',
               '2017-07-11', '2017-11-16', '2019-05-22', '2025-04-21'],
              dtype='datetime64[ns]', freq=None)


Adding the FX block removes 30 additional dates from the Core Master
Table (5680 → 5650 observations).

A cluster of missing FX observations occurs in August 2008.
These observations are not forward-filled or reconstructed.

The final dataset therefore retains only dates with directly observed
inputs across all feature blocks.

## Feature Set Frozen Before Label Construction

The final POC03 feature set contains 13 features across five
economic blocks:

- Rates Pressure: 4
- Price & Concentration: 3
- Credit Resilience: 2
- Volatility Resilience: 2
- FX / Cross-Asset Response: 2

# Details
Rates Pressure
- LevelZ252
- Momentum20BP
-  ShockZ60
-  DistanceHighVolAdj60

Market Response
-  SPYMomentum20
-  SPYDistanceHigh60
-  EqualWeightRelative20
-  HYOASLevel
-  HYOASChange20BP
-  VIXLevel
-  VIXChange20

Cross-Asset / FX
-  DXYChange20
-  USDJPYChange20

DXYChange20 and USDJPYChange20 are calculated on each series'
native valid observation calendar before alignment.

No future equity outcome has been constructed or inspected at the
time of feature selection.

From this point forward, no additional features, lookback windows,
or transformations will be added based on subsequent FAE_20D results.

Any later feature additions will be explicitly labeled as
exploratory follow-up research.

# Creating Label: FAE_20D

In [49]:
# ============================================================
# POC03 — Primary Label
# Forward Adverse Excursion over next 20 SPY trading days
# ============================================================

spy_label = price[["SPY"]].copy()

future_returns = pd.concat(
    {
        k: spy_label["SPY"].shift(-k) / spy_label["SPY"] - 1
        for k in range(1, 21)
    },
    axis=1
)

spy_label["FAE_20D"] = future_returns.min(
    axis=1,
    skipna=False
)
# 完整20个future trading days才有label 否则FAE_20D = NaN

In [50]:
# QA
print("Label shape:", spy_label.shape)

print("\nMissing FAE_20D:")
print(spy_label["FAE_20D"].isna().sum())

print("\nFAE summary:")
display(spy_label["FAE_20D"].describe())

print("\nLast 25 observations:")
display(spy_label.tail(25))

Label shape: (5724, 2)

Missing FAE_20D:
20

FAE summary:


,FAE_20D
count,5704.000000
mean,-0.024911
std,0.036743
min,-0.309868
25%,-0.035654
50%,-0.014783
75%,-0.002824
max,0.090603



Last 25 observations:


Ticker,SPY,FAE_20D
Date,,
2026-08-28,767.444275,-0.019887
2026-08-31,765.149963,-0.016948
2026-09-01,759.893066,-0.010147
2026-09-02,763.264648,-0.014520
2026-09-03,771.254822,-0.024729
2026-09-04,768.282227,NaN
2026-09-08,764.062683,NaN
2026-09-09,760.511536,NaN
2026-09-10,755.952820,NaN


In [51]:
# Another Sanity check
print(
    "FAE > 0 count:",
    (spy_label["FAE_20D"] > 0).sum()
)

print(
    "FAE = 0 count:",
    (spy_label["FAE_20D"] == 0).sum()
)

print(
    "FAE < 0 count:",
    (spy_label["FAE_20D"] < 0).sum()
)

FAE > 0 count: 1039
FAE = 0 count: 10
FAE < 0 count: 4655


In [52]:
 # manual spot check
 check_date = "2020-02-18"

loc = spy_label.index.get_loc(check_date)

current_price = spy_label.loc[check_date, "SPY"]

next20 = spy_label["SPY"].iloc[
    loc + 1 : loc + 21
]

manual_fae = (
    next20.min() / current_price - 1
)

print("Date:", check_date)
print("Current SPY:", current_price)
print("Lowest SPY next 20 trading days:", next20.min())

print(
    "Manual FAE:",
    manual_fae
)

print(
    "Calculated FAE:",
    spy_label.loc[check_date, "FAE_20D"]
)

Date: 2020-02-18
Current SPY: 306.1756591796875
Lowest SPY next 20 trading days: 218.08639526367188
Manual FAE: -0.28770825268091627
Calculated FAE: -0.28770825268091627


# Build final modeling dataset

In [53]:
model_data = (
    master_final
    .join(
        spy_label[["FAE_20D"]],
        how="left"
    )
)

feature_cols = [
    "LevelZ252",
    "Momentum20BP",
    "ShockZ60",
    "DistanceHighVolAdj60",

    "SPYMomentum20",
    "SPYDistanceHigh60",
    "EqualWeightRelative20",

    "HYOASLevel",
    "HYOASChange20BP",

    "VIXLevel",
    "VIXChange20",

    "DXYChange20",
    "USDJPYChange20"
]

analysis_data = model_data[
    feature_cols + ["FAE_20D"]
].dropna()

print("Model data shape:", model_data.shape)
print("Analysis data shape:", analysis_data.shape)

print("\nDate range:")
print(
    analysis_data.index.min(),
    "to",
    analysis_data.index.max()
)

print("\nMissing:")
print(analysis_data.isna().sum())

print("\nFAE summary:")
display(analysis_data["FAE_20D"].describe())

Model data shape: (5651, 15)
Analysis data shape: (5380, 14)

Date range:
2005-01-05 00:00:00 to 2026-09-03 00:00:00

Missing:
LevelZ252                0
Momentum20BP             0
ShockZ60                 0
DistanceHighVolAdj60     0
SPYMomentum20            0
SPYDistanceHigh60        0
EqualWeightRelative20    0
HYOASLevel               0
HYOASChange20BP          0
VIXLevel                 0
VIXChange20              0
DXYChange20              0
USDJPYChange20           0
FAE_20D                  0
dtype: int64

FAE summary:


,FAE_20D
count,5380.000000
mean,-0.025161
std,0.037368
min,-0.309868
25%,-0.035766
50%,-0.014748
75%,-0.002778
max,0.090603


In [54]:
rate_cut = analysis_data["LevelZ252"].median()

resilience_cut = analysis_data[
    "EqualWeightRelative20"
].median()

print("Rate median:", rate_cut)
print("Resilience median:", resilience_cut)

Rate median: 0.11191728276720908
Resilience median: -0.0005655884829577351


In [55]:
test = analysis_data.copy()

test["RateState"] = np.where(
    test["LevelZ252"] >= rate_cut,
    "High Rates",
    "Low Rates"
)

test["ResilienceState"] = np.where(
    test["EqualWeightRelative20"] < resilience_cut,
    "Weak Resilience",
    "Strong Resilience"
)

result_2x2 = (
    test
    .groupby(
        ["RateState", "ResilienceState"]
    )["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(result_2x2)

Count      Mean    Median
RateState  ResilienceState                             
High Rates Strong Resilience   1385 -0.023136 -0.015159
           Weak Resilience     1305 -0.022002 -0.014507
Low Rates  Strong Resilience   1305 -0.024813 -0.013510
           Weak Resilience     1385 -0.030491 -0.015984

# DEV / CONFIRM research

In [56]:
dev = analysis_data.loc[
    "2005-01-01":"2022-12-31"
].copy()

confirm = analysis_data.loc[
    "2023-01-01":"2025-12-31"
].copy()

live = analysis_data.loc[
    "2026-01-01":
].copy()

print("DEV:", dev.shape, dev.index.min(), dev.index.max())
print("CONFIRM:", confirm.shape, confirm.index.min(), confirm.index.max())
print("LIVE:", live.shape, live.index.min(), live.index.max())

DEV: (4465, 14) 2005-01-05 00:00:00 2022-12-30 00:00:00
CONFIRM: (746, 14) 2023-01-03 00:00:00 2025-12-31 00:00:00
LIVE: (169, 14) 2026-01-02 00:00:00 2026-09-03 00:00:00


In [57]:
dev["RateQ"] = pd.qcut(
    dev["LevelZ252"],
    5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

rate_q = (
    dev.groupby("RateQ", observed=True)["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(rate_q)

,Count,Mean,Median
RateQ,,,
Q1,893,-0.028642,-0.017399
Q2,893,-0.025976,-0.012190
Q3,893,-0.027829,-0.014508
Q4,893,-0.021578,-0.013530
Q5,893,-0.027506,-0.018857


In [58]:
dev["ResilienceQ"] = pd.qcut(
    dev["EqualWeightRelative20"],
    5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

resilience_q = (
    dev.groupby(
        "ResilienceQ",
        observed=True
    )["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(resilience_q)

,Count,Mean,Median
ResilienceQ,,,
Q1,893,-0.036755,-0.018773
Q2,893,-0.023871,-0.015149
Q3,893,-0.020282,-0.014004
Q4,893,-0.023573,-0.012630
Q5,893,-0.027049,-0.015059


这里记住方向：
- RateQ5 = LevelZ252最高
- ResilienceQ1 = RSP相对SPY最弱
- FAE越负 = future downside越严重

In [59]:
rate_features = [
    "LevelZ252",
    "Momentum20BP",
    "ShockZ60",
    "DistanceHighVolAdj60"
]

rate_tables = {}

for feature in rate_features:

    temp = dev[[feature, "FAE_20D"]].copy()

    temp["Q"] = pd.qcut(
        temp[feature],
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )

    table = (
        temp.groupby("Q", observed=True)["FAE_20D"]
        .agg(
            Count="count",
            Mean="mean",
            Median="median"
        )
    )

    rate_tables[feature] = table

    print("\n====================")
    print(feature)
    print("====================")
    display(table)


LevelZ252


,Count,Mean,Median
Q,,,
Q1,893,-0.028642,-0.017399
Q2,893,-0.025976,-0.012190
Q3,893,-0.027829,-0.014508
Q4,893,-0.021578,-0.013530
Q5,893,-0.027506,-0.018857



Momentum20BP


,Count,Mean,Median
Q,,,
Q1,896,-0.032809,-0.017048
Q2,907,-0.020473,-0.010607
Q3,895,-0.023559,-0.014710
Q4,877,-0.022990,-0.014290
Q5,890,-0.031734,-0.020968



ShockZ60


,Count,Mean,Median
Q,,,
Q1,893,-0.028745,-0.017207
Q2,893,-0.026324,-0.015296
Q3,893,-0.023190,-0.012953
Q4,893,-0.026208,-0.014811
Q5,893,-0.027064,-0.015143



DistanceHighVolAdj60


,Count,Mean,Median
Q,,,
Q1,893,-0.028666,-0.013459
Q2,893,-0.025827,-0.012025
Q3,893,-0.026330,-0.015415
Q4,893,-0.024955,-0.018123
Q5,893,-0.025754,-0.015765


## Rate feature conclusion: Among the four preregistered Treasury-rate dimensions, no feature shows a stable monotonic relationship with FAE_20D. Momentum20BP shows the most notable pattern, with worse downside in both extreme falling-rate and extreme rising-rate regimes. This observation is exploratory and will not be used to redefine the feature set.

In [60]:
# Check EqualWeightRelative20 Q1 cutoff threshold
resilience_q20_cut = dev["EqualWeightRelative20"].quantile(0.20)

print(
    "DEV EqualWeightRelative20 Q20 cutoff:",
    resilience_q20_cut
)

DEV EqualWeightRelative20 Q20 cutoff: -0.008071641597834333


过去约20个交易日里，RSP相对SPY少涨/多跌超过约0.81个百分点，我们把它定义成 Weak Resilience (历史上最弱的20% equal-weight relative performance)

In [61]:
#OOS sample testing
confirm_test = confirm.copy()

confirm_test["WeakResilience"] = (
    confirm_test["EqualWeightRelative20"]
    < resilience_q20_cut
)

confirm_result = (
    confirm_test
    .groupby("WeakResilience")["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(confirm_result)

print("\nWeak Resilience frequency:")
print(
    confirm_test["WeakResilience"]
    .value_counts(normalize=True)
)

,Count,Mean,Median
WeakResilience,,,
False,377,-0.022223,-0.014864
True,369,-0.016515,-0.011213



Weak Resilience frequency:
WeakResilience
False    0.505362
True     0.494638
Name: proportion, dtype: float64


# 第一段研究结果：
① 高10Y本身不是可靠减仓signal。
② RSP/SPY突然弱也不能直接当成未来20D大跌signal。
③ 固定历史threshold在结构变化的市场里很危险。

# next step: 在 DEV 2005–2022 中，看看 Market Response 的各个组成部分，哪些跟未来 FAE_20D 有值得继续研究的结构

In [62]:
response_features = [
    # Price / Concentration
    "SPYMomentum20",
    "SPYDistanceHigh60",
    "EqualWeightRelative20",

    # Credit
    "HYOASLevel",
    "HYOASChange20BP",

    # Volatility
    "VIXLevel",
    "VIXChange20",

    # FX / Cross-Asset
    "DXYChange20",
    "USDJPYChange20"
]

response_tables = {}

for feature in response_features:

    temp = dev[[feature, "FAE_20D"]].copy()

    temp["Q"] = pd.qcut(
        temp[feature],
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
        duplicates="drop"
    )

    table = (
        temp.groupby("Q", observed=True)["FAE_20D"]
        .agg(
            Count="count",
            Mean="mean",
            Median="median"
        )
    )

    response_tables[feature] = table

    print("\n====================")
    print(feature)
    print("====================")
    display(table)


SPYMomentum20


,Count,Mean,Median
Q,,,
Q1,893,-0.039480,-0.024575
Q2,893,-0.025019,-0.014746
Q3,893,-0.021668,-0.013170
Q4,893,-0.020487,-0.012555
Q5,893,-0.024878,-0.014742



SPYDistanceHigh60


,Count,Mean,Median
Q,,,
Q1,893,-0.044865,-0.028484
Q2,893,-0.025994,-0.016230
Q3,893,-0.021718,-0.014037
Q4,893,-0.019185,-0.011101
Q5,893,-0.019769,-0.012555



EqualWeightRelative20


...

,Count,Mean,Median
Q,,,
Q1,893,-0.023476,-0.012219
Q2,893,-0.021313,-0.013011
Q3,893,-0.022423,-0.013752
Q4,893,-0.026480,-0.015843
Q5,893,-0.037840,-0.024305



DXYChange20


,Count,Mean,Median
Q,,,
Q1,893,-0.024463,-0.017010
Q2,893,-0.022592,-0.012219
Q3,893,-0.024535,-0.014188
Q4,893,-0.025707,-0.015107
Q5,893,-0.034233,-0.016230



USDJPYChange20


,Count,Mean,Median
Q,,,
Q1,893,-0.033205,-0.019116
Q2,893,-0.021258,-0.012015
Q3,893,-0.026963,-0.015557
Q4,893,-0.026084,-0.015250
Q5,893,-0.024021,-0.013881


## DEV Quintile Diagnostics — Market Response Features

### Purpose

Before constructing a composite **Market Resilience** signal, examine each frozen market-response feature separately in the DEV sample (2005–2022).

Each feature is sorted into five quintiles:

- **Q1** = lowest 20% of observations
- **Q5** = highest 20% of observations

The direction of "weakness" depends on the economic meaning of each feature.

| Feature | Q1 Represents | Q5 Represents | Intuitively Weaker Side |
|---|---|---|---|
| `SPYMomentum20` | Weakest 20D SPY return | Strongest 20D SPY return | Q1 |
| `SPYDistanceHigh60` | Furthest below 60D high | Closest to 60D high | Q1 |
| `EqualWeightRelative20` | RSP weakest vs SPY | RSP strongest vs SPY | Q1 |
| `HYOASLevel` | Lowest HY spread | Highest HY spread | Q5 |
| `HYOASChange20BP` | Largest spread tightening | Largest spread widening | Q5 |
| `VIXLevel` | Lowest VIX | Highest VIX | Q5 |
| `VIXChange20` | Largest VIX decline | Largest VIX increase | Q5 |
| `DXYChange20` | Largest USD weakening | Largest USD strengthening | No prior assumption |
| `USDJPYChange20` | Largest JPY appreciation | Largest JPY depreciation | No prior assumption |

---

### DEV Results — Main Observations

#### 1. Equity Price Damage

`SPYMomentum20` and `SPYDistanceHigh60` show strong relationships with future downside.

- Weakest `SPYMomentum20` quintile:
  - Mean FAE20 ≈ **-3.95%**
- Furthest-from-high `SPYDistanceHigh60` quintile:
  - Mean FAE20 ≈ **-4.49%**

This suggests that once equity price damage is already visible, downside risk tends to persist.

However, these variables may primarily capture **risk persistence / volatility clustering**, rather than provide an early warning before equity weakness appears.

---

#### 2. Equal-Weight Relative Weakness

`EqualWeightRelative20` showed substantial downside in DEV Q1:

- Q1 Mean FAE20 ≈ **-3.68%**

However, the DEV-defined Q20 threshold failed to confirm in the 2023–2025 CONFIRM sample and reversed direction.

Therefore:

> Equal-weight underperformance alone should not be treated as a robust standalone downside signal.

It may describe market concentration without necessarily implying weak overall market resilience.

---

#### 3. Credit Conditions

Credit produced some of the strongest economically interesting patterns.

`HYOASLevel`:

- Q1 Mean FAE20 ≈ **-2.05%**
- Q5 Mean FAE20 ≈ **-3.96%**

`HYOASChange20BP`:

- Q3 Mean FAE20 ≈ **-1.89%**
- Q5 Mean FAE20 ≈ **-4.24%**

The particularly poor outcome in Q5 of `HYOASChange20BP` suggests that **rapid credit-spread deterioration** may contain useful risk information.

This is especially interesting because spread widening may occur before severe headline-equity damage.

---

#### 4. Volatility

`VIXLevel` shows the clearest monotonic relationship in the DEV diagnostics:

Mean FAE20 deteriorates approximately:

**-1.52% → -1.89% → -2.39% → -3.10% → -4.25%**

as VIX moves from Q1 to Q5.

`VIXChange20` also shows substantially worse downside in Q5:

- Q5 Mean FAE20 ≈ **-3.78%**

Interpretation:

> Elevated or rapidly rising implied volatility is associated with greater subsequent downside.

However, VIX may partly be a **confirmation / persistence indicator** because equity stress may already be underway when VIX becomes elevated.

---

#### 5. FX / Cross-Asset

FX relationships are less stable.

`DXYChange20`:
- Q5 Mean FAE20 ≈ **-3.42%**
- Median differences are much smaller.

`USDJPYChange20`:
- Q1 Mean FAE20 ≈ **-3.32%**
- The relationship is not monotonic across quintiles.

Therefore, FX variables remain useful cross-asset context, but current DEV evidence is insufficient to treat either as a primary resilience signal.

---

### Working Interpretation

The DEV diagnostics suggest three conceptually different stages of market stress:

**1. Early / Cross-Asset Deterioration**
- Credit spreads begin widening
- Potentially before major equity damage

**2. Volatility Repricing**
- VIX rises as risk perception changes

**3. Equity Damage / Confirmation**
- SPY momentum deteriorates
- SPY moves materially below recent highs

A possible risk-transmission framework is therefore:

**Macro Pressure → Credit Deterioration → Volatility Repricing → Equity Price Damage**

This is an economic hypothesis / organizing framework, **not yet a demonstrated causal sequence**.

---

### Key Research Implication

The most useful practical signal may not be:

> "Is SPY already weak?"

but instead:

> **"Is credit deteriorating while SPY and VIX still appear relatively resilient?"**

Such a state could potentially provide an **early-warning signal** before obvious headline-equity stress.

The next experiment should therefore test whether extreme credit deterioration (`HYOASChange20BP`) contains incremental downside information when:

- SPY has not yet suffered major price damage, and
- VIX has not yet entered a high-stress regime.

Thresholds must be defined using DEV only and frozen before evaluating future outcomes or the CONFIRM sample.

In [63]:
dev_thresholds = {
    "HYOASChange20BP_Q80":
        dev["HYOASChange20BP"].quantile(0.80),

    "SPYDistanceHigh60_Q20":
        dev["SPYDistanceHigh60"].quantile(0.20),

    "VIXLevel_Q80":
        dev["VIXLevel"].quantile(0.80)
}

dev_thresholds

{'HYOASChange20BP_Q80': np.float64(33.999999999999986),
 'SPYDistanceHigh60_Q20': np.float64(-0.05644807433510568),
 'VIXLevel_Q80': np.float64(24.520000457763672)}

In [64]:
early_warning = dev.copy()

early_warning["CreditEarlyWarning"] = (
    (early_warning["HYOASChange20BP"] > 34.0)
    & (early_warning["SPYDistanceHigh60"] >= -0.05644808439418734)
    & (early_warning["VIXLevel"] <= 24.520000457763672)
)

early_warning_result = (
    early_warning
    .groupby("CreditEarlyWarning")["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(early_warning_result)

print("\nEarly Warning frequency:")
print(
    early_warning["CreditEarlyWarning"]
    .value_counts(normalize=True)
)

,Count,Mean,Median
CreditEarlyWarning,,,
False,4154,-0.025836,-0.014454
True,311,-0.032587,-0.026931



Early Warning frequency:
CreditEarlyWarning
False    0.930347
True     0.069653
Name: proportion, dtype: float64


In [65]:
# ============================================================
# Frozen DEV-developed Credit Early Warning Rule
# ============================================================

CREDIT_WIDENING_CUTOFF = 34.0
PRICE_DAMAGE_CUTOFF = -0.05644808439418734
VIX_STRESS_CUTOFF = 24.520000457763672

confirm_ew = confirm.copy()

confirm_ew["CreditEarlyWarning"] = (
    (confirm_ew["HYOASChange20BP"] > CREDIT_WIDENING_CUTOFF)
    & (confirm_ew["SPYDistanceHigh60"] >= PRICE_DAMAGE_CUTOFF)
    & (confirm_ew["VIXLevel"] <= VIX_STRESS_CUTOFF)
)

confirm_ew_result = (
    confirm_ew
    .groupby("CreditEarlyWarning")["FAE_20D"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median"
    )
)

display(confirm_ew_result)

print("\nCONFIRM Early Warning frequency:")
print(
    confirm_ew["CreditEarlyWarning"]
    .value_counts(normalize=True)
)

,Count,Mean,Median
CreditEarlyWarning,,,
False,715,-0.019687,-0.013001
True,31,-0.012773,-0.006235



CONFIRM Early Warning frequency:
CreditEarlyWarning
False    0.958445
True     0.041555
Name: proportion, dtype: float64


## POC03-A Primary Result — Not Supported

### Original Hypothesis

Under elevated Treasury-rate pressure, weakening market resilience was expected to be associated with more negative 20-day Forward Adverse Excursion (`FAE_20D`).

### Result

The primary hypothesis was **not supported**.

Four preregistered Treasury-rate features did not show a stable monotonic relationship with future 20-day downside in DEV.

Several market-response variables showed strong DEV relationships, particularly:

- equity price damage,
- HY credit deterioration,
- elevated/rising VIX.

However, candidate signals developed from DEV failed to generalize.

**Equal-weight weakness**
- showed worse downside in DEV,
- but reversed direction in the 2023–2025 CONFIRM sample.

**Credit Early Warning rule**
- DEV Mean FAE20: **-3.26%**
- DEV Normal Mean FAE20: **-2.58%**

Using the frozen DEV thresholds in CONFIRM:

- Early Warning Mean FAE20: **-1.28%**
- Normal Mean FAE20: **-1.97%**

Median results also reversed direction.

### Conclusion

Neither extreme equal-weight underperformance nor the DEV-developed Credit Early Warning rule provided stable out-of-sample evidence of greater future equity downside.

The evidence therefore does not support using these fixed historical rules as standalone risk-sizing signals.

### Research Lesson

Strong economic intuition and attractive in-sample relationships are insufficient.

The DEV → CONFIRM reversals suggest substantial **non-stationarity / regime dependence** in cross-asset risk relationships.

This motivates a new exploratory question:

> Under what market regimes do credit, volatility, breadth, and rates signals contain useful information about future equity downside?

Any investigation of this question will be treated as a **new exploratory study**, rather than a modification or rescue of the original POC03 hypothesis.

In [66]:
# QA ① Overlap Robustness：解决 FAE20 的19/20重叠
# ============================================================
# POC03-A Final Robustness Check 1
# Non-overlapping 20D FAE — 20 Offset Analysis
# ============================================================

import pandas as pd
import numpy as np

CREDIT_WIDENING_CUTOFF = 34.0
PRICE_DAMAGE_CUTOFF = -0.05644808439418734
VIX_STRESS_CUTOFF = 24.520000457763672


def add_credit_early_warning(df):
    out = df.copy()

    out["CreditEarlyWarning"] = (
        (out["HYOASChange20BP"] > CREDIT_WIDENING_CUTOFF)
        & (out["SPYDistanceHigh60"] >= PRICE_DAMAGE_CUTOFF)
        & (out["VIXLevel"] <= VIX_STRESS_CUTOFF)
    )

    return out


def offset_robustness(df, sample_name):

    temp = add_credit_early_warning(df)

    results = []

    for offset in range(20):

        # Every 20th observation -> approximately non-overlapping FAE windows
        sub = temp.iloc[offset::20].copy()

        normal = sub.loc[
            ~sub["CreditEarlyWarning"], "FAE_20D"
        ]

        warning = sub.loc[
            sub["CreditEarlyWarning"], "FAE_20D"
        ]

        results.append({
            "Sample": sample_name,
            "Offset": offset,
            "Normal_N": len(normal),
            "Warning_N": len(warning),
            "Normal_Mean": normal.mean(),
            "Warning_Mean": warning.mean(),
            "Mean_Difference":
                warning.mean() - normal.mean(),
            "Normal_Median": normal.median(),
            "Warning_Median": warning.median(),
            "Median_Difference":
                warning.median() - normal.median()
        })

    return pd.DataFrame(results)


dev_offsets = offset_robustness(dev, "DEV")
confirm_offsets = offset_robustness(confirm, "CONFIRM")

offset_results = pd.concat(
    [dev_offsets, confirm_offsets],
    ignore_index=True
)

display(offset_results)

,Sample,Offset,Normal_N,Warning_N,Normal_Mean,Warning_Mean,Mean_Difference,Normal_Median,Warning_Median,Median_Difference
0,DEV,0,209,15,-0.027380,-0.029603,-0.002223,-0.016251,-0.033020,-0.016769
1,DEV,1,212,12,-0.026246,-0.037812,-0.011566,-0.017058,-0.038602,-0.021544
2,DEV,2,212,12,-0.026669,-0.040284,-0.013615,-0.015873,-0.036563,-0.020689
3,DEV,3,206,18,-0.026154,-0.031942,-0.005787,-0.014972,-0.026881,-0.011908
4,DEV,4,209,15,-0.026097,-0.033318,-0.007222,-0.014798,-0.029033,-0.014235
5,DEV,5,207,16,-0.026211,-0.034262,-0.008051,-0.014606,-0.034370,-0.019764
6,DEV,6,203,20,-0.025283,-0.032407,-0.007124,-0.012375,-0.029649,-0.017274
7,DEV,7,203,20,-0.023272,-0.038415,-0.015143,-0.013962,-0.025753,-0.011791
8,DEV,8,205,18,-0.024454,-0.032802,-0.008348,-0.014507,-0.023281,-0.008774
9,DEV,9,211,12,-0.025844,-0.031910,-0.006066,-0.014804,-0.025594,-0.010789


In [67]:
# ============================================================
# Directional Stability Across 20 Offsets
# ============================================================

offset_summary = (
    offset_results
    .groupby("Sample")
    .agg(
        Offsets=("Offset", "count"),

        MeanDiff_Avg=("Mean_Difference", "mean"),
        MeanDiff_Median=("Mean_Difference", "median"),

        MedianDiff_Avg=("Median_Difference", "mean"),
        MedianDiff_Median=("Median_Difference", "median"),

        Mean_Support_Count=(
            "Mean_Difference",
            lambda x: (x < 0).sum()
        ),

        Median_Support_Count=(
            "Median_Difference",
            lambda x: (x < 0).sum()
        )
    )
)

display(offset_summary)

,Offsets,MeanDiff_Avg,MeanDiff_Median,MedianDiff_Avg,MedianDiff_Median,Mean_Support_Count,Median_Support_Count
Sample,,,,,,,
CONFIRM,20,0.008229,0.010948,0.002536,0.005999,5,5
DEV,20,-0.006813,-0.007173,-0.011531,-0.013072,18,17


In [68]:
# QA 2
# ============================================================
# POC03-A Final Robustness Check 2
# Credit Early Warning Episode Analysis
# ============================================================

def build_warning_episodes(df, sample_name):

    temp = add_credit_early_warning(df).copy()

    # New episode begins whenever True follows False
    temp["EpisodeStart"] = (
        temp["CreditEarlyWarning"]
        & ~temp["CreditEarlyWarning"].shift(1, fill_value=False)
    )

    temp["EpisodeID"] = temp["EpisodeStart"].cumsum()

    warnings = temp[temp["CreditEarlyWarning"]].copy()

    episodes = (
        warnings
        .groupby("EpisodeID")
        .agg(
            StartDate=("CreditEarlyWarning", lambda x: x.index.min()),
            EndDate=("CreditEarlyWarning", lambda x: x.index.max()),
            WarningDays=("CreditEarlyWarning", "size"),

            Start_HYOASChange20BP=(
                "HYOASChange20BP", "first"
            ),
            Max_HYOASChange20BP=(
                "HYOASChange20BP", "max"
            ),

            Start_VIX=(
                "VIXLevel", "first"
            ),
            Max_VIX=(
                "VIXLevel", "max"
            ),

            Start_SPYDistanceHigh60=(
                "SPYDistanceHigh60", "first"
            ),

            Start_FAE20=(
                "FAE_20D", "first"
            )
        )
        .reset_index()
    )

    episodes.insert(0, "Sample", sample_name)

    return episodes


dev_episodes = build_warning_episodes(dev, "DEV")
confirm_episodes = build_warning_episodes(confirm, "CONFIRM")

episodes = pd.concat(
    [dev_episodes, confirm_episodes],
    ignore_index=True
)

print("DEV warning days:",
      add_credit_early_warning(dev)["CreditEarlyWarning"].sum())

print("DEV episodes:", len(dev_episodes))

print("\nCONFIRM warning days:",
      add_credit_early_warning(confirm)["CreditEarlyWarning"].sum())

print("CONFIRM episodes:", len(confirm_episodes))

display(episodes)

DEV warning days: 311
DEV episodes: 75

CONFIRM warning days: 31
CONFIRM episodes: 8


,Sample,EpisodeID,StartDate,EndDate,WarningDays,Start_HYOASChange20BP,Max_HYOASChange20BP,Start_VIX,Max_VIX,Start_SPYDistanceHigh60,Start_FAE20
0,DEV,1,2005-01-24,2005-01-24,1,38.0,38.0,14.650000,14.650000,-0.039634,0.002832
1,DEV,2,2005-03-29,2005-04-14,13,45.0,86.0,14.490000,14.530000,-0.047254,-0.023427
2,DEV,3,2005-04-19,2005-04-19,1,92.0,92.0,14.960000,14.960000,-0.056411,-0.013950
3,DEV,4,2005-04-21,2005-04-25,3,63.0,69.0,14.410000,15.380000,-0.051505,-0.015602
4,DEV,5,2005-04-27,2005-04-27,1,66.0,66.0,14.870000,14.870000,-0.054448,-0.012538
...,...,...,...,...,...,...,...,...,...,...,...
78,CONFIRM,4,2023-10-10,2023-10-17,6,41.0,48.0,17.030001,19.320000,-0.047439,-0.054909
79,CONFIRM,5,2024-04-18,2024-04-18,1,34.0,34.0,18.000000,18.000000,-0.045205,-0.008728
80,CONFIRM,6,2024-08-12,2024-08-14,3,39.0,46.0,20.709999,20.709999,-0.055925,0.013295
81,CONFIRM,7,2025-10-10,2025-10-10,1,39.0,39.0,21.660000,21.660000,-0.029846,0.011669


In [69]:
episode_summary = (
    episodes
    .groupby("Sample")
    .agg(
        Episodes=("EpisodeID", "count"),
        MedianWarningDays=("WarningDays", "median"),
        MeanWarningDays=("WarningDays", "mean"),
        MeanStartFAE20=("Start_FAE20", "mean"),
        MedianStartFAE20=("Start_FAE20", "median"),
        WorstStartFAE20=("Start_FAE20", "min")
    )
)

display(episode_summary)

,Episodes,MedianWarningDays,MeanWarningDays,MeanStartFAE20,MedianStartFAE20,WorstStartFAE20
Sample,,,,,,
CONFIRM,8,2.0,3.875000,-0.006884,-0.004351,-0.054909
DEV,75,2.0,4.146667,-0.022484,-0.015797,-0.156074


# Final Research Record

# POC03-A — Final Research Record

## Rates Pressure × Market Resilience
### Does Cross-Asset Stress Predict Future Equity Downside?

**Status: CLOSED**

---

## 1. Research Objective

The objective of POC03-A was to test whether elevated Treasury-rate pressure,
combined with weakening market resilience, could identify periods of greater
future equity downside risk.

The practical objective was **risk sizing / avoiding large losses**, rather than
predicting short-term SPY direction.

---

## 2. Primary Hypothesis

The preregistered directional hypothesis was:

> **High Rates Pressure + Weakening Market Resilience  
> → More Negative 20-Day Forward Adverse Excursion**

The primary outcome variable was:

**FAE_20D — 20-Day Forward Adverse Excursion**

Defined as the minimum SPY return observed over the next 20 trading days:

FAE_20D(t) = min(SPY(t+k) / SPY(t) - 1), for k = 1,...,20

More negative values represent greater subsequent downside.

The 20-trading-day horizon was frozen before outcome analysis.

---

## 3. Dataset and Sample Design

Final analysis sample:

- 5,380 complete observations
- 2005-01-05 to 2026-09-03

Research periods:

- **DEV:** 2005-01-05 to 2022-12-30
- **CONFIRM:** 2023-01-03 to 2025-12-31
- **LIVE:** 2026-01-02 onward

The 2026 sample was treated as a live case rather than a fully untouched
out-of-sample period because contemporary market conditions contributed to
the original research motivation.

Features were calculated on each source's native calendar before cross-asset
alignment.

No forward filling was used.

---

## 4. Frozen Feature Set

### Rates Pressure

1. `LevelZ252`
   - Current 10Y Treasury yield relative to its prior 252-observation history.

2. `Momentum20BP`
   - 20-observation change in the 10Y Treasury yield, measured in basis points.

3. `ShockZ60`
   - Standardized daily Treasury-yield change relative to the prior 60 observations.

4. `DistanceHighVolAdj60`
   - Volatility-adjusted distance from the rolling 60-observation Treasury-yield high.

### Equity Price / Concentration

5. `SPYMomentum20`
6. `SPYDistanceHigh60`
7. `EqualWeightRelative20`

### Credit

8. `HYOASLevel`
9. `HYOASChange20BP`

### Volatility

10. `VIXLevel`
11. `VIXChange20`

### FX / Cross-Asset

12. `DXYChange20`
13. `USDJPYChange20`

SPY price itself was retained only as the source for constructing the future
outcome label and was not used as a predictive feature.

---

## 5. DEV Findings — Rates Pressure

None of the four Treasury-rate dimensions showed a stable monotonic
relationship with future FAE_20D.

`Momentum20BP` showed the most notable nonlinear pattern:

- extreme rate increases were associated with worse future downside,
- but extreme rate decreases were also associated with worse downside.

This suggested that large rate movements may characterize unstable regimes,
rather than providing a simple directional equity-risk signal.

No rate feature was selected as a standalone validated predictor.

---

## 6. DEV Findings — Market Response

Several market-response variables showed strong relationships with future
downside in DEV.

### Equity Price Damage

Weak SPY momentum and large distance from recent highs were associated with
substantially worse subsequent FAE_20D.

These relationships were interpreted primarily as potential
**risk persistence / damage confirmation**, rather than necessarily early
warning signals.

### Credit

Rapid HY OAS widening showed a particularly strong DEV relationship.

The highest quintile of `HYOASChange20BP` had:

- Mean FAE_20D ≈ **-4.24%**
- Median FAE_20D ≈ **-2.89%**

### Volatility

`VIXLevel` showed the clearest monotonic DEV relationship.

Mean FAE_20D deteriorated approximately from:

**-1.52% → -1.89% → -2.39% → -3.10% → -4.25%**

from the lowest to highest VIX quintile.

However, elevated VIX may partly represent an already-active risk regime
rather than a leading signal.

---

## 7. Equal-Weight Weakness Test

DEV analysis suggested that extreme equal-weight underperformance might
identify greater future downside.

A DEV-defined 20th-percentile threshold was frozen:

`EqualWeightRelative20 < -0.81%`

In the 2023–2025 CONFIRM sample, approximately 49.5% of observations fell
below this historical threshold, demonstrating substantial distribution shift.

More importantly, the expected downside relationship reversed:

- Not Weak Mean FAE_20D: **-2.22%**
- Weak Mean FAE_20D: **-1.65%**

Median results also reversed.

Therefore, the DEV relationship did not generalize out of sample.

---

## 8. DEV-Developed Credit Early Warning Rule

Based on DEV diagnostics, a secondary early-warning rule was developed.

This was explicitly a **DEV-developed rule**, not part of the original
preregistered hypothesis.

DEV thresholds were frozen as:

- `HYOASChange20BP > +34 bp`
- `SPYDistanceHigh60 >= -5.64%`
- `VIXLevel <= 24.52`

Economic interpretation:

> Credit is deteriorating rapidly while headline equity price and implied
> volatility have not yet entered severe stress.

### DEV Result

Early Warning observations:

- Count: **311**
- Frequency: **6.97%**
- Mean FAE_20D: **-3.26%**
- Median FAE_20D: **-2.69%**

Normal observations:

- Mean FAE_20D: **-2.58%**
- Median FAE_20D: **-1.45%**

The DEV sample therefore supported the candidate early-warning rule.

---

## 9. CONFIRM Result

The exact frozen DEV rule was applied to 2023–2025 without recalibrating
thresholds.

Early Warning:

- Count: **31**
- Frequency: **4.16%**
- Mean FAE_20D: **-1.28%**
- Median FAE_20D: **-0.62%**

Normal:

- Count: **715**
- Mean FAE_20D: **-1.97%**
- Median FAE_20D: **-1.30%**

The relationship reversed direction.

The DEV-developed Credit Early Warning signal therefore **failed to
generalize out of sample**.

---

## 10. Overlap Robustness

Because adjacent FAE_20D labels share up to 19 future trading days,
a 20-offset non-overlapping robustness analysis was performed.

### DEV

- Mean relationship supported in **18 / 20 offsets**
- Median relationship supported in **17 / 20 offsets**
- Average Mean Difference: **-0.68 percentage points**
- Average Median Difference: **-1.15 percentage points**

### CONFIRM

- Mean relationship supported in only **5 / 20 offsets**
- Median relationship supported in only **5 / 20 offsets**
- Average Mean Difference: **+0.82 percentage points**
- Average Median Difference: **+0.25 percentage points**

Therefore, the DEV relationship was not simply an artifact of overlapping
20-day labels.

The main failure was **out-of-sample instability**, not label overlap.

CONFIRM offset estimates should nevertheless be interpreted cautiously because
individual offsets contained very few warning observations.

---

## 11. Independent Episode Analysis

Consecutive Early Warning observations were collapsed into single episodes.

### DEV

- Warning days: **311**
- Independent episodes: **75**
- Median episode duration: **2 days**
- Mean duration: **4.15 days**
- Mean FAE_20D from first warning: **-2.25%**
- Median FAE_20D from first warning: **-1.58%**
- Worst episode FAE_20D: **-15.61%**

### CONFIRM

- Warning days: **31**
- Independent episodes: **8**
- Median episode duration: **2 days**
- Mean duration: **3.88 days**
- Mean FAE_20D from first warning: **-0.69%**
- Median FAE_20D from first warning: **-0.44%**
- Worst episode FAE_20D: **-5.49%**

The DEV effect therefore persisted after collapsing repeated daily warnings
into independent episodes.

However, it again failed to generalize to the CONFIRM period.

The small number of CONFIRM episodes limits precision, but the available
out-of-sample evidence does not support the signal.

---

## 12. Final Conclusion

### Primary hypothesis: NOT SUPPORTED

The study found no stable evidence that:

> **Elevated Treasury-rate pressure combined with weakening market resilience
> reliably predicts greater 20-day equity downside.**

Several economically intuitive relationships appeared strong in DEV,
particularly:

- equity price damage,
- credit-spread deterioration,
- elevated volatility.

However, candidate early-warning signals derived from these relationships did
not survive the untouched 2023–2025 CONFIRM period.

The failure remained after:

- freezing DEV thresholds,
- preserving the 20-day outcome horizon,
- testing non-overlapping samples,
- and collapsing repeated warnings into independent episodes.

---

## 13. Practical Investment Implication

The results do **not** support mechanically reducing equity risk based on:

- elevated Treasury yields alone,
- extreme equal-weight underperformance alone,
- or the tested fixed Credit Early Warning rule.

A signal that appears economically convincing and historically strong can
still fail when the market regime changes.

For practical risk management, signal stability across regimes matters more
than in-sample strength.

---

## 14. Main Research Lesson

The most important finding from POC03-A is not a trading signal.

It is evidence of **non-stationarity / regime dependence**.

Relationships that were strong and economically intuitive during 2005–2022
did not remain stable during 2023–2025.

This suggests a new research question:

> **Under what market regimes do rates, credit, volatility, and market-internal
> signals contain useful information about future equity downside?**

This question is outside the scope of POC03-A.

Any investigation will be treated as a separate exploratory study rather than
a modification or rescue of the original hypothesis.

---

## Research Status

**POC03-A CLOSED**

The hypothesis was tested, failed out of sample, subjected to overlap and
episode-level robustness checks, and accepted as unsupported without
post-hoc threshold optimization.